In [0]:
dbutils.widgets.text("retail_env", "")

ENV = dbutils.widgets.get("retail_env").strip().lower()

if ENV not in ["dev", "test", "prod"]:
    raise ValueError(f"Invalid or missing environment: {ENV}")

CATALOG = "retail"

BRONZE_SCHEMA = f"{ENV}_bronze"
SILVER_SCHEMA = f"{ENV}_silver"
GOLD_SCHEMA = f"{ENV}_gold"

print(f"Current environment: {ENV}")
print(f"Catalog: {CATALOG}")
print(f"Bronze: {BRONZE_SCHEMA}")
print(f"Silver: {SILVER_SCHEMA}")
print(f"Gold: {GOLD_SCHEMA}")

Current environment: dev
Catalog: retail
Bronze: dev_bronze
Silver: dev_silver
Gold: dev_gold


## Datenbereinigung und Transformation – Silver Layer

Der untere Code liest zunächst die Tabelle `purchase_orders` aus der Bronze-Schicht (`retail.dev_bronze.purchase_orders`) ein.

Anschließend werden die Daten bereinigt und transformiert, um eine strukturierte und qualitativ verbesserte Tabelle für die Silver-Schicht zu erstellen.

Dabei werden folgende Schritte durchgeführt:

- **Duplikate entfernen:** Mit `dropDuplicates()` werden doppelte Datensätze anhand der Spalten `ean13`, `ean5` und `PO` entfernt.
- **Datentypen anpassen:** Die Produktidentifikatoren `EAN13` und `EAN5` werden in String umgewandelt. Die Purchase-Order-ID wird als Integer gespeichert.
- **Zeitstempel konvertieren:** Der Unix-Timestamp aus der Spalte `datetime` wird mit `from_unixtime()` in ein lesbares Timestamp-Format umgewandelt.
- **Preis bereinigen:** Mit `trim()` und `regexp_replace()` werden Leerzeichen und Sonderzeichen aus der Preisspalte entfernt. Anschließend wird der Preis in `decimal(18,2)` umgewandelt.
- **String-Bereinigung:** Die Spalten `product_name`, `product_unit`, `purchaser` und `supplier` werden von überflüssigen Leerzeichen bereinigt und als String gespeichert.
- **Menge konvertieren:** Die Spalte `quantity` wird in den Datentyp Integer umgewandelt.
- **Nicht benötigte Spalten entfernen:** Die Spalten `password` und `datetime` werden entfernt, da sie nicht mehr benötigt werden. Das konvertierte Datum bleibt in `promotion_datetime` erhalten.

Zum Schluss werden die Anzahl der Datensätze vor und nach der Bereinigung ausgegeben und die ersten fünf Datensätze des bereinigten DataFrames mit `display(silver_df.limit(5))` angezeigt.

**Ergebnis:** Es entsteht ein bereinigter und typisierter DataFrame, der als Grundlage für die Speicherung in der Silver-Schicht dient.

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    regexp_replace,
    expr,
    from_unixtime   
)
from pyspark.sql import functions as F


bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.purchase_orders")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["ean13", "ean5", "PO"])

silver_df = (df_deduplicated

    # Identifiers -> String
    .withColumn("ean13", F.col("EAN13").cast("string"))\
    .withColumn("ean5", F.col("EAN5").cast("string"))\
  
    # Purchase Order ID

    .withColumn("purchase_order", F.col("PO").cast("int"))\
    # Unix Timestamp -> Timestamp
     .withColumn("promotion_datetime", F.from_unixtime(F.col("datetime")).cast("timestamp"))\

    # Password NICHT übernehmen

    # "$ 39" -> 39.00
    .withColumn(
    "price",
    regexp_replace(
        trim(col("price")),
        r"[^0-9.-]",
        ""
    ).cast("decimal(18,2)")
)\

    # Strings bereinigen
    .withColumn("product_name", trim(F.col("product_name")).cast("string"))\
    .withColumn("product_unit", trim(F.col("product_unit")).cast("string"))\
    .withColumn("purchaser", trim(F.col("purchaser")).cast("string"))\

    # Quantity
    .withColumn("quantity", trim(F.col("quantity")).cast("int"))\

    # Supplier
    .withColumn("supplier", trim(F.col("supplier")).cast("string"))

    # drop password and datetime
    .drop("password", "datetime")

)

print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {silver_df.count()}")

display(silver_df.limit(5))

Datensätze vor Bereinigung: 3973 
Datensätze nach Bereinigung (ohne Duplikate): 3973


ean13,ean5,PO,price,product_name,product_unit,purchaser,quantity,supplier,purchase_order,promotion_datetime
2198122550193,55019,934213542,39.00,Details About Mogitech G920 Xbox Driving Force Racing Wheel For Xbox One And Pc (941000121),pcs,"JANUS, PAUL C",15,Mogitech,934213542,2019-08-01T00:30:48.000Z
2198122550745,55074,934214594,17.00,Mogitech - Harmony 950 Universal Remote - Black,pcs,"CARMEN, ZACHARY C",5,Mogitech,934214594,2019-08-01T00:53:59.000Z
2198122550745,55074,934213362,24.00,Mogitech - Harmony 950 Universal Remote - Black,pcs,"PHILLIPS, DEBORAH A",15,Mogitech,934213362,2019-08-01T01:05:55.000Z
2198122550038,55003,934213097,28.00,Mogitech Ultrathin Touch Mouse T630 for Windows 8 Touch Gestures,pcs,"HARRIS, THOMAS H",10,Mogitech,934213097,2019-08-01T02:12:03.000Z
2198122549911,54991,934212803,30.00,Mogitech Keys-To-Go Ultra-Portable Bluetooth Keyboard for Android and Windows,pcs,"BRADLEY, VALENA M",5,Mogitech,934212803,2019-08-01T03:25:19.000Z



### 1. Die Pflichtspalten (Harte Filter)

Diese Spalten dürfen keine Null-Werte enthalten. Wenn sie leer sind, fliegt die Zeile raus oder geht in die Quarantäne:

 *  **PO** (**oder purchase_order**– je nachdem, welche Spalte Ihre **primäre Bestellnummer** ist).
 * **ean13** (**Ohne den Artikelcode** ist die Zeile unbrauchbar).


### 2. Die Spalten für Finanzmathematik (Ersetzen durch Zahlenwerte)

* quantity: Bei Null ersetzen durch 0. (Verhindert Berechnungsfehler).
* price: Bei Null ersetzen durch 0.00 oder über einen Lookup aus dim_products. (Verhindert, dass der Gesamtumsatz zu Null wird).


### 3. Die Text- und Gruppierungsspalten (Ersetzen durch Standard-Texte)

* **product_name**: Bei Null ersetzen durch "Unbekanntes Produkt".
* **product_unit**: Bei Null ersetzen durch "Stück".
* **purchaser**: Bei Null ersetzen durch "Nicht zugewiesen".
* **supplier**: Bei Null ersetzen durch "Unbekannter Lieferant".

In [0]:
from pyspark.sql.functions import col, coalesce, lit

# Filterung der Pflichtfelder und Behandlung aller Null-Werte
silver_po_final = (silver_df
    # 1. Zeilen ohne Bestellnummer oder Artikelnummer direkt herausfiltern
    .filter(col("PO").isNotNull() & col("ean13").isNotNull())
    
    # 2. Null-Werte durch logische Standards ersetzen
    .select(
        col("PO").alias("purchase_order_id"),
        col("purchase_order"),
        col("ean13"),
        coalesce(col("ean5"), lit("")).alias("ean5"),
        
        # Mengen und Preise vor Null-Werten schützen
        coalesce(col("quantity"), lit(0)).alias("quantity"),
        coalesce(col("price"), lit(0.00)).alias("unit_price"),
        
        # Text-Spalten lesbar machen
        coalesce(col("product_name"), lit("Unbekanntes Produkt")).alias("product_name"),
        coalesce(col("product_unit"), lit("Stück")).alias("order_unit"),
        coalesce(col("purchaser"), lit("System-Einkäufer")).alias("purchaser_name"),
        coalesce(col("supplier"), lit("Unbekannter Lieferant")).alias("supplier_name"),
        
        col("promotion_datetime")
    )
   
)

# Ergebnis in Databricks anzeigen
display(silver_po_final.limit(3))

print(f"Datensätze vor Bereinigung: { silver_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {silver_po_final.count()}")

silver_po_final.printSchema()

purchase_order_id,purchase_order,ean13,ean5,quantity,unit_price,product_name,order_unit,purchaser_name,supplier_name,promotion_datetime
934213542,934213542,2198122550193,55019,15,39.0,Details About Mogitech G920 Xbox Driving Force Racing Wheel For Xbox One And Pc (941000121),pcs,"JANUS, PAUL C",Mogitech,2019-08-01T00:30:48.000Z
934214594,934214594,2198122550745,55074,5,17.0,Mogitech - Harmony 950 Universal Remote - Black,pcs,"CARMEN, ZACHARY C",Mogitech,2019-08-01T00:53:59.000Z
934213362,934213362,2198122550745,55074,15,24.0,Mogitech - Harmony 950 Universal Remote - Black,pcs,"PHILLIPS, DEBORAH A",Mogitech,2019-08-01T01:05:55.000Z


Datensätze vor Bereinigung: 3973 
Datensätze nach Bereinigung (ohne Duplikate): 3973
root
 |-- purchase_order_id: long (nullable = true)
 |-- purchase_order: integer (nullable = true)
 |-- ean13: string (nullable = true)
 |-- ean5: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: double (nullable = false)
 |-- product_name: string (nullable = false)
 |-- order_unit: string (nullable = false)
 |-- purchaser_name: string (nullable = false)
 |-- supplier_name: string (nullable = false)
 |-- promotion_datetime: timestamp (nullable = true)



## Das Silver Layer kurz zusammengefasst:

*  **Datenqualität garantiert**: Alle Duplikate, Rechenfehler (durch Null-Werte) und Textfehler wurden dauerhaft beseitigt.

* **Strukturierte Wahrheit**: Die Daten wurden standardisiert, sinnvoll umbenannt und in korrekte Datentypen (wie Decimal und Timestamp) gecastet.

* **Sicher und Bereit**: Sensible Daten (Passwörter) wurden entfernt. Die Tabelle ist nun die perfekte, absturzsichere Startrampe für alle Berichte im Gold Layer.

In [0]:
#spark.sql("CREATE SCHEMA IF NOT EXISTS retail.dev_silver")


StatementMeta(, 7c9af660-688a-489a-895d-930399ca1988, 7, Finished, Available, Finished, False)

DataFrame[]

In [0]:
silver_po_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.purchase_orders")



## Datenbereinigung und Transformation – Active Promotions (Silver Layer)

Der untere Code liest zunächst die Tabelle `active_promotions` aus der Bronze-Schicht (`retail.dev_bronze.active_promotions`) ein.

Anschließend werden die Daten bereinigt, standardisiert und in ein einheitliches Zielschema überführt.

Dabei werden folgende Schritte durchgeführt:

- **Duplikate entfernen:** Mit `dropDuplicates()` werden doppelte Datensätze anhand der Spalten `promo_id`, `promo_datetime` und `promo_item` entfernt.
- **Zeitstempel-Konvertierung:** Die Hilfsfunktion `convert_timestamp()` vereinheitlicht unterschiedliche Zeitstempelformate. Dabei werden Unix-Timestamps in Sekunden und Millisekunden sowie reguläre Datums- und Zeitangaben berücksichtigt.
- **Identifikatoren standardisieren:** Die Spalten `promo_id`, `promo_item` und `promo_customer` werden in `promotion_id`, `product_id` und `customer_id` umbenannt und als String gespeichert.
- **Kampagnenzeitraum:** Die Spalten `promo_began` und `deadline` werden in `campaign_start` und `campaign_end` umgewandelt.
- **Einlösungszeitpunkt:** Der Zeitstempel `promo_datetime` wird in `redemption_timestamp` überführt.
- **Numerische Datentypen:** Die Spalten `eligible`, `units_required` und `promo_qty` werden in Integer umgewandelt.
- **Rabattstandardisierung:** Der Rabattwert `promo_disc` wird als `decimal(5,4)` gespeichert und in `percentage_discount_rate` übernommen.
- **Zielspalten auswählen:** Mit `select()` werden ausschließlich die benötigten und standardisierten Spalten übernommen.

Zum Schluss werden die Anzahl der Datensätze vor und nach der Transformation ausgegeben. Mit `display(silver_df.limit(5))` werden die ersten fünf Datensätze zur Kontrolle angezeigt.

**Ergebnis:** Es entsteht ein bereinigter und standardisierter DataFrame mit einheitlichen Spaltennamen und Datentypen als Grundlage für die Silver-Schicht.

In [0]:

from pyspark.sql import functions as F

# 1. Laden der echten CSV-Tabelle aus Bronze
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.active_promotions")

# 2. Duplikate entfernen
df_deduplicated = bronze_df.dropDuplicates(
    ["promo_id", "promo_datetime", "promo_item"]
)

# 3. Hilfsfunktion zur Zeitstempel-Konvertierung
def convert_timestamp(column_name):

    value = F.trim(F.col(column_name))

    return (
        F.when(
            value.rlike(r"^\d{10}$"),
            F.to_timestamp(
                F.from_unixtime(value.cast("long"))
            )
        )
        .when(
            value.rlike(r"^\d{13}$"),
            F.to_timestamp(
                F.from_unixtime(
                    (value.cast("long") / 1000).cast("long")
                )
            )
        )
        .otherwise(
            F.expr(
                f"try_cast(trim(`{column_name}`) AS TIMESTAMP)"
            )
        )
    )

# 4. Transformation & technische Standardisierung
silver_df = (
    df_deduplicated

    # Identifikatoren
    .withColumn("promotion_id", F.col("promo_id").cast("string"))
    .withColumn("product_id", F.trim(F.col("promo_item")).cast("string"))
    .withColumn("customer_id", F.trim(F.col("promo_customer")).cast("string"))

    # Zeitstempel: Unix Sekunden, Millisekunden oder normales Datum
    .withColumn("campaign_start", convert_timestamp("promo_began"))
    .withColumn("campaign_end", convert_timestamp("deadline"))
    .withColumn("redemption_timestamp", convert_timestamp("promo_datetime"))

    # Numerische Datentypen
    .withColumn("is_eligible_flag", F.col("eligible").cast("int"))
    .withColumn("min_units_required", F.col("units_required").cast("int"))
    .withColumn("discounted_quantity", F.col("promo_qty").cast("int"))

    # Rabattsatz
    .withColumn(
        "percentage_discount_rate",
        F.col("promo_disc").cast("decimal(5,4)")
    )

    # Nur saubere Zielspalten behalten
    .select(
        "promotion_id",
        "product_id",
        "customer_id",
        "campaign_start",
        "campaign_end",
        "redemption_timestamp",
        "is_eligible_flag",
        "min_units_required",
        "discounted_quantity",
        "percentage_discount_rate"
    )
)

# 5. Ergebnisse kontrollieren
print(f"Datensätze vor Bereinigung: {bronze_df.count()}")
print(f"Datensätze nach Transformation: {silver_df.count()}")

display(silver_df.limit(5))


Datensätze vor Bereinigung: 423
Datensätze nach Transformation: 423


promotion_id,product_id,customer_id,campaign_start,campaign_end,redemption_timestamp,is_eligible_flag,min_units_required,discounted_quantity,percentage_discount_rate
1,AVqVGUFCv8e3D1O-ldFF,10294348,2019-11-11T21:25:49.000Z,null,2019-11-11T21:25:49.000Z,0,5,2,0.0500
0,AVpfMVD-ilAPnD_xW6bu,10393805,2019-08-01T21:29:19.000Z,null,2019-08-01T21:29:19.000Z,0,3,1,0.0300
0,AVpfMVD-ilAPnD_xW6bu,10393805,2019-08-01T21:29:19.000Z,null,2019-08-13T10:26:24.000Z,0,3,1,0.0300
0,AVpfMVD-ilAPnD_xW6bu,10393805,2019-08-01T21:29:19.000Z,null,2019-08-27T04:34:40.000Z,1,3,2,0.0300
0,AVpfMVD-ilAPnD_xW6bu,10393805,2019-08-01T21:29:19.000Z,null,2019-09-25T21:19:31.000Z,1,3,3,0.0300


# Datenqualität & Null-Wert-Behandlung (active_promotion)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Lückenlose Absicherung der transformierten Marketing- und Rabattdaten gegen `NULL`- und `NaN`-Werte. Dieser Schritt garantiert eine mathematisch absolut stabile Basis für nachgelagerte Umsatz- und Effizienzanalysen im Gold Layer.

---

## 1. Pflichtspalten (Harter Filter)
Diese Spalten bilden das logische Fundament jeder Promotion. Datensätze, die diese Kriterien verletzen, werden rigoros aus dem Datenfluss entfernt, da sie analytisch nicht zugeordnet werden können.

* **`promotion_id`**: Ohne eine gültige Kampagnen-ID kann keine Zuordnung zu Marketing-Aktivitäten erfolgen.
* **`product_id`**: Ohne die Artikel-ID (EAN) ist es unmöglich zu bestimmen, welches Produkt rabattiert wurde.

---

## 2. Text-Standardisierung (Business-Platzhalter)
Unter Verwendung der zentralen Funktion `clean_string_nulls` werden alle Textfelder systematisch bereinigt. Technische Fragmente wie `"null"`, `"NaN"`, `"nan"` oder komplett leere Zeichenketten `""` werden durch vordefinierte Standardwerte ersetzt:

* **`customer_id`**: Wenn keine Kunden-ID übergeben wurde (z. B. bei anonymen Einkäufen ohne Kundenkarte an der Kasse), wird das Feld standardmäßig auf **`"GAST"`** gesetzt.

---

## 3. Numerischer Schutz & Flags (Schutz vor NULL und NaN)
Mengen, Bedingungen und Rabattsätze werden durch die Kombination aus `coalesce()` und `nanvl()` sowohl gegen klassische Datenbank-`NULL`-Werte als auch gegen mathematische `NaN`-Laufzeitfehler abgesichert:

* **`is_eligible_flag`**: Fehlt der Berechtigungsstatus, wird er standardmäßig auf **`0`** (Nicht berechtigt) gesetzt.
* **`min_units_required`**: Wenn der Schwellenwert für die Mindestmenge fehlt, wird er auf den neutralen Standardwert **`1`** gesetzt.
* **`discounted_quantity`**: Fehlt die Angabe zur rabattierten Menge, wird sie auf **`0`** gesetzt.
* **`percentage_discount_rate`**: Der prozentuale Rabattsatz wird bei Fehlern mathematisch auf **`0.0000`** (0% Rabatt) gezwungen. Dies verhindert fatale `Null-Pointer-Exceptions` bei der späteren Umsatzberechnung im Gold Layer (`Menge * Preis * Rabatt`).

---

## 4. Zeitstempel-Erhalt
* **`campaign_start`**, **`campaign_end`**, **`redemption_timestamp`**: Diese Zeitstempel werden für zeitraumbezogene Validierungen im Gold Layer unverändert mitgeführt.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, nanvl, when, trim 

# Eine wiederverwendbare Logik für alle Textspalten
def clean_string_nulls(column_name, default_text):
    return when(
        (col(column_name).isNull()) | 
        (trim(col(column_name)) == "NaN") | 
        (trim(col(column_name)) == "nan") | 
        (trim(col(column_name)) == "null") |
        (trim(col(column_name)) == ""), 
        lit(default_text)
    ).otherwise(trim(col(column_name)))

# Finale Qualitätsbereinigung für die Marketing-Aktionen
silver_promotions_final = (silver_df
    # 1. PFLICHTSPALTEN FILTERN: Ohne Kampagnen-ID oder Artikel-ID ist die Zeile ungültig
    .filter(
        col("promotion_id").isNotNull() & (trim(col("promotion_id")) != "NaN") & (trim(col("promotion_id")) != "nan") &
        col("product_id").isNotNull() & (trim(col("product_id")) != "NaN") & (trim(col("product_id")) != "nan")
    )
    
    # 2. FINALE NULL & NaN BEHANDLUNG FÜR TEXT UND ZAHLEN
    .select(
        col("promotion_id"),
        col("product_id"),
        
        # TEXT-SPALTEN: Schützt vor NULL, "NaN", "null" und Leerzeichen
        clean_string_nulls("customer_id", "GAST").alias("customer_id"),
        
        # NUMERISCHE SPALTEN / FLAGS: Schützt vor echtem NULL und mathematischem NaN
        coalesce(nanvl(col("is_eligible_flag"), lit(0)), lit(0)).cast("int").alias("is_eligible_flag"),
        coalesce(nanvl(col("min_units_required"), lit(1)), lit(1)).cast("int").alias("min_units_required"),
        coalesce(nanvl(col("discounted_quantity"), lit(0)), lit(0)).cast("int").alias("discounted_quantity"),
        
        # Hochpräziser Rabattsatz (Decimal): Schutz vor NULL und NaN
        coalesce(nanvl(col("percentage_discount_rate"), lit(0.0000)), lit(0.0000)).cast("decimal(5,4)").alias("percentage_discount_rate"),
        
        # Zeitstempel für Gültigkeiten und Einlösungen mitführen
        col("campaign_start"),
        col("campaign_end"),
        col("redemption_timestamp")
    )
)

# 3. In die finale Silver Delta-Tabelle schreiben
silver_promotions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.active_promotions")

print(f"Promotion-Daten erfolgreich im Silver Layer gespeichert! Anzahl Datensätze: {silver_promotions_final.count()}")
display(silver_promotions_final.limit(5))


Promotion-Daten erfolgreich im Silver Layer gespeichert! Anzahl Datensätze: 423


promotion_id,product_id,customer_id,is_eligible_flag,min_units_required,discounted_quantity,percentage_discount_rate,campaign_start,campaign_end,redemption_timestamp
1,AVqVGUFCv8e3D1O-ldFF,10294348,0,5,2,0.0500,2019-11-11T21:25:49.000Z,null,2019-11-11T21:25:49.000Z
0,AVpfMVD-ilAPnD_xW6bu,10393805,0,3,1,0.0300,2019-08-01T21:29:19.000Z,null,2019-08-01T21:29:19.000Z
0,AVpfMVD-ilAPnD_xW6bu,10393805,0,3,1,0.0300,2019-08-01T21:29:19.000Z,null,2019-08-13T10:26:24.000Z
0,AVpfMVD-ilAPnD_xW6bu,10393805,1,3,2,0.0300,2019-08-01T21:29:19.000Z,null,2019-08-27T04:34:40.000Z
0,AVpfMVD-ilAPnD_xW6bu,10393805,1,3,3,0.0300,2019-08-01T21:29:19.000Z,null,2019-09-25T21:19:31.000Z


Wenn Sie diese Daten später im Gold Layer für eine **geografische Landkarte** (über latitude/longitude) nutzen oder Berichte nach state (Bundesland) gruppieren, wird Ihr Dashboard niemals abstürzen oder hässliche, leere Zeilen anzeigen. Alle Daten weisen nun eine garantierte 100%ige Struktur auf.


## Datenbereinigung und Transformation – Sales Orders (Silver Layer)

Der untere Code liest zunächst die Tabelle `sales_orders` aus der Bronze-Schicht (`retail.dev_bronze.sales_orders`) ein.

Anschließend werden die Daten bereinigt, Datentypen angepasst und die Spalten in ein einheitliches Zielschema überführt.

Dabei werden folgende Schritte durchgeführt:

- **Duplikate entfernen:** Mit `dropDuplicates()` werden doppelte Datensätze anhand der Spalten `order_number` und `order_datetime` entfernt.
- **Order Number:** Die Bestellnummer wird von überflüssigen Leerzeichen bereinigt und in `BIGINT` umgewandelt. Mit `try_cast()` werden ungültige Werte abgefangen.
- **Customer ID:** Die Kunden-ID wird von überflüssigen Leerzeichen bereinigt und als String gespeichert.
- **Customer Name:** Der Kundenname wird ebenfalls bereinigt und als String gespeichert.
- **Number of Line Items:** Die Anzahl der Bestellpositionen wird in den Datentyp Integer umgewandelt.
- **Zeitstempel-Konvertierung:** Der Unix-Timestamp aus `order_datetime` wird mit `timestamp_seconds()` in ein lesbares Timestamp-Format umgewandelt und in der neuen Spalte `orderdate` gespeichert.
- **Zielspalten auswählen:** Mit `select()` werden ausschließlich die benötigten Spalten in das Silver-Zielschema übernommen.

Zum Schluss werden die Anzahl der Datensätze vor und nach der Bereinigung ausgegeben. Mit `display(orders_silver.limit(5))` werden die ersten fünf Datensätze zur Kontrolle angezeigt.

**Ergebnis:** Es entsteht ein bereinigter und standardisierter DataFrame mit einheitlichen Datentypen und einer konvertierten Bestellzeit als Grundlage für die Silver-Schicht.

In [0]:

from pyspark.sql import functions as F

# 1. Bronze laden
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.sales_orders")

# 2. Duplikate entfernen
df_deduplicated = bronze_df.dropDuplicates(
    ["order_number", "order_datetime"]
)

# 3. Silver Transformation
orders_silver = (
    df_deduplicated

    # Order Number
    .withColumn(
        "order_number",
        F.expr("try_cast(trim(order_number) AS BIGINT)")
    )

    # Customer ID
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id")).cast("string")
    )

    # Customer Name
    .withColumn(
        "customer_name",
        F.trim(F.col("customer_name")).cast("string")
    )

    # Number of Line Items
    .withColumn(
        "number_of_line_items",
        F.expr("try_cast(trim(number_of_line_items) AS INT)")
    )

    # Unix Timestamp in Sekunden
    .withColumn(
        "orderdate",
        F.expr("""
            timestamp_seconds(
                try_cast(trim(order_datetime) AS BIGINT)
            )
        """)
    )

    # Nur benötigte Spalten behalten
    .select(
        "order_number",
        "customer_id",
        "customer_name",
        "number_of_line_items",
        "orderdate"
    )
)

# 4. Ergebnisse kontrollieren
print(f"Datensätze vor Bereinigung: {bronze_df.count()}")
print(f"Datensätze nach Bereinigung: {orders_silver.count()}")

display(orders_silver.limit(5))


Datensätze vor Bereinigung: 4074
Datensätze nach Bereinigung: 4074


order_number,customer_id,customer_name,number_of_line_items,orderdate
317568014,19476252,"otbda , outside the box digital agency ,",3,2019-08-01T02:47:43.000Z
317568015,4401099,denevi digital imaging,1,2019-08-01T03:27:15.000Z
317568016,14939501,rpm optoelectronics,2,2019-08-01T04:07:10.000Z
317568017,3072597,"non typical, inc.",3,2019-08-01T05:01:48.000Z
317568018,2732808,als deli,2,2019-08-01T05:03:17.000Z


In [0]:
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.company_employees")

print(bronze_df.printSchema())

root
 |-- employee_id: string (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- region: string (nullable = true)
 |-- employee_key: string (nullable = true)
 |-- active_record: string (nullable = true)
 |-- active_record_start: string (nullable = true)
 |-- active_record_end: string (nullable = true)

None


# Datenqualität & Null-Wert-Behandlung (sales_orders)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Sicherung einer fehlerfreien und konsistenten Grundlage für Umsatz-, Warenkorb- und Kundenkanal-Analysen im Gold Layer durch die gezielte Beseitigung und Standardisierung von `NULL`-Werten.

---

### 1. Pflichtspalten (Harter Filter)
Diese Spalten dürfen **niemals** `NULL` oder leer sein. Datensätze, die diese Kriterien verletzen, werden rigoros aus dem Datenfluss entfernt.

* **`order_number`**: Ein Verkaufsauftrag ohne eindeutige Belegnummer ist kaufmännisch und analytisch ungültig.
* **`customer_id`**: Ohne eine gültige Kunden-ID ist ein späterer Join mit den Kundenstammdaten (`dim_customers`) unmöglich.

### 2. Zahlen absichern (Numerischer Schutz)
Fehlende numerische Werte werden durch mathematisch neutrale Standardwerte ersetzt, um Berechnungsfehler (z. B. `Null-Pointer-Exceptions`) bei Aggregationen zu verhindern.

* **`number_of_line_items`**: Wenn die Anzahl der Positionen im Auftrag `NULL` ist, wird sie standardmäßig auf **`0`** gesetzt. Dies sichert die Berechnung der durchschnittlichen Warenkorbgröße.

### 3. Texte standardisieren (Business-Platzhalter)
Leere Textfelder werden durch sprechende Platzhalter ersetzt, um unschöne Lücken oder kryptische `NULL`-Einträge in Berichtsfiltern und BI-Dashboards zu vermeiden.

* **`customer_name`**: Fehlt der Kundenname (z. B. durch technische Fehler bei Express-Checkouts im Onlineshop), wird er durch den Standardwert **`"Gastbesteller"`** oder **`"Unbekannter Kunde"`** ersetzt.

### 4. Zeitstempel-Validierung
* **`orderdate`**: Ein fehlendes Bestelldatum führt zu Fehlern in Zeitreihenanalysen. Falls dieser Wert `NULL` ist, wird er auf den Verarbeitungszeitstempel des Systems (`current_timestamp()`) gesetzt, um die zeitliche Einordnung zu gewährleisten.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, current_timestamp, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem orders_silver DataFrame
silver_orders_final = (orders_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder customer_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("customer_id").isNotNull() & 
        (trim(col("customer_id")) != "") &
        (trim(col("customer_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), 
        clean_string_nulls("customer_id", "UNBEKANNT").alias("customer_id"),
        
        # Text standardisieren unter Verwendung der clean_string_nulls Funktion
        clean_string_nulls("customer_name", "Gastbesteller").alias("customer_name"),
        
        # Zahlen absichern (NULL durch 0 ersetzen)
        coalesce(col("number_of_line_items"), lit(0)).alias("number_of_line_items"),
        
        # Zeitstempel absichern (Falls NULL, aktuellen System-Zeitstempel nutzen)
        coalesce(col("orderdate"), current_timestamp()).alias("order_date")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_orders_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.sales_orders")

# Protokollierung der Datensätze
print(f"Datensätze in orders_silver: {orders_silver.count()}")
print(f"Datensätze nach finaler Qualitätsbereinigung: {silver_orders_final.count()}")

display(silver_orders_final.limit(5))


Datensätze in orders_silver: 4074
Datensätze nach finaler Qualitätsbereinigung: 4074


sales_order_id,customer_id,customer_name,number_of_line_items,order_date
317568014,19476252,"otbda , outside the box digital agency ,",3,2019-08-01T02:47:43.000Z
317568015,4401099,denevi digital imaging,1,2019-08-01T03:27:15.000Z
317568016,14939501,rpm optoelectronics,2,2019-08-01T04:07:10.000Z
317568017,3072597,"non typical, inc.",3,2019-08-01T05:01:48.000Z
317568018,2732808,als deli,2,2019-08-01T05:03:17.000Z


# Silver Layer – Sales Orders Cleaning

## Ziel

In diesem Schritt werden die Sales Orders aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Datentypen**, der **Bereinigung von Kundeninformationen**, der **Konvertierung des Bestelldatums** sowie der **Auswahl der relevanten Spalten**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Order Number standardisieren

Die `order_number` wird in den Datentyp `Long` konvertiert:

* `order_number` → `Long`

Dadurch wird sichergestellt, dass die Bestellnummer in einem einheitlichen numerischen Datentyp vorliegt.

### 2. Customer ID bereinigen

Die `customer_id` wird zunächst von führenden und nachfolgenden Leerzeichen bereinigt und anschließend als String gespeichert:

* `customer_id` → `String`

Die Verwendung eines Strings ist für Kunden-IDs geeignet, da diese als Identifikatoren und nicht als numerische Werte behandelt werden.

### 3. Customer Name bereinigen

Der Kundenname wird mit `trim()` bereinigt und als String gespeichert:

* `customer_name` → `String`

Dadurch werden überflüssige Leerzeichen am Anfang und Ende des Wertes entfernt.

### 4. Anzahl der Bestellpositionen standardisieren

Das Feld `number_of_line_items` wird in einen Integer-Datentyp konvertiert:

* `number_of_line_items` → `Integer`

Damit kann die Anzahl der Bestellpositionen später für numerische Analysen verwendet werden.

### 5. Bestelldatum standardisieren

Das ursprüngliche Feld `order_datetime` enthält einen Unix-Timestamp.

Dieser wird zunächst in einen numerischen `Long`-Wert konvertiert und anschließend in einen Spark Timestamp umgewandelt:

* `order_datetime` → `orderdate`
* Unix Timestamp → `Timestamp`

Dadurch steht das Bestelldatum in einem standardisierten Format für zeitbasierte Analysen zur Verfügung.

### 6. Relevante Spalten auswählen

Nach den Transformationen werden nur die für den Silver Layer benötigten Spalten übernommen:

* `order_number`
* `customer_id`
* `customer_name`
* `number_of_line_items`
* `orderdate`

Das ursprüngliche Feld `order_datetime` wird dadurch nicht weitergeführt.

## Datenqualität überprüfen

Nach der Transformation werden die ersten fünf Datensätze des bereinigten DataFrames angezeigt:

```python
display(orders_silver.limit(5))


In [0]:
from pyspark.sql.functions import col, explode
from pyspark.sql import functions as F

items_silver = df_deduplicated.select(
    F.col("order_number"),
    explode(F.col("ordered_products")).alias("product")
)

items_silver = (items_silver

     .withColumn("order_number", 
         F.col("order_number")
         .cast("long")
     )

     .withColumn("product_id",
          F.col("product.id")
          .cast("string")
     )

     .withColumn("product_name",
           F.col("product.name")
           .cast("string")
     )

     .withColumn("currency",
           F.col("product.curr")
           .cast("string")
     )

     .withColumn("price",
           F.col("product.price")
           .cast("decimal(18,2)")
     )
 

      .withColumn("quantity", 
            F.col("product.qty")
            .cast("int")
      )
       

     .withColumn("unit", 
            F.col("product.unit")
            .cast("string")
     )
)


items_silver=items_silver.select(
                  "order_number",
                  "product_id",
                  "product_name",
                  "currency",
                  "price",
                  "quantity",
                  "unit"
)
display(items_silver.limit(5))

order_number,product_id,product_name,currency,price,quantity,unit
317568017,AVpiMIyE1cnluZ0-K0TA,Elpine - Rear View Camera - Black,USD,60.00,1,pcs
317568017,AVz5wc0H-jtxr-f30F6_,Zamaha - AVENTAGE 7.2-Ch. 4K Ultra HD A/V Home Theater Receiver - Black,USD,149.00,9,pcs
317568017,AVpe7vER1cnluZ0-aJu7,Mogitech Keys-To-Go Ultra-Portable Bluetooth Keyboard for Android and Windows,USD,618.00,3,pcs
317568028,AVpjedgc1cnluZ0-W4NI,Rony MEXM100BT 160W RMS Marine CD Receiver with Bluetooth (Black) and SiriusXM Ready,USD,287.00,2,pcs
317568028,AVpfMVD-ilAPnD_xW6bu,Rony - BC-TRX Battery Charger - Black,USD,27.00,1,pcs


# Datenqualität & Null-Wert-Behandlung (sales_order_items)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Transformation von geschachtelten Bestelldaten (*Explode*) in ein flaches, konsistentes Positionsdaten-Modell. Es sichert die mathematische Korrektheit aller Einzelpositionen für spätere Umsatz- und Margenberechnungen im Gold Layer ab.

---

### 1. Pflichtspalten (Harter Filter)
Diese Spalten verknüpfen den Artikel mit dem Auftrag. Fehlen sie, ist die Tabellenzeile unbrauchbar und wird ausgefiltert.

* **`order_number`**: Ohne Zuordnung zu einem Hauptauftrag verliert die Position ihre Gültigkeit.
* **`product_id`**: Jede Positionszeile muss zwingend einem konkreten Produkt zugeordnet sein, um spätere Stammdaten-Joins zu ermöglichen.

### 2. Zahlen absichern (Numerischer Schutz)
Mengen und Preise dürfen niemals `NULL` sein, um Berechnungsfehler bei der Ermittlung des Gesamtumsatzes zu verhindern.

* **`quantity`**: Wenn die Verkaufsmenge `NULL` ist, wird sie standardmäßig auf **`0`** gesetzt.
* **`price`**: Wenn kein Einzelpreis übergeben wurde, wird er mit **`0.00`** initialisiert.

### 3. Texte standardisieren (Business-Platzhalter)
Die Funktion `clean_string_nulls` bereinigt Textspalten von technischen Fehlern wie `"null"` oder `"NaN"`.

* **`product_name`**: Fehlende Namen werden durch den Platzhalter **`"Unbekannter Artikel"`** ersetzt.
* **`currency`**: Fehlt die Währung, wird der Standardwert **`"EUR"`** eingetragen.
* **`unit`**: Eine fehlende Verkaufseinheit wird standardmäßig als **`"Stück"`** deklariert.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F

# 1. Null-Wert-Behandlung basierend auf Ihrem items_silver DataFrame
silver_items_final = (items_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder product_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("product_id").isNotNull() & 
        (trim(col("product_id")) != "") &
        (trim(col("product_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), # Einheitlicher Name für den Key
        clean_string_nulls("product_id", "UNBEKANNT").alias("product_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("product_name", "Unbekannter Artikel").alias("product_name"),
        clean_string_nulls("currency", "EUR").alias("currency"),
        clean_string_nulls("unit", "Stück").alias("order_unit"),
        
        # Zahlen & Preise absichern
        coalesce(col("price"), lit(0.00)).alias("unit_price"),
        coalesce(col("quantity"), lit(0)).alias("quantity_ordered")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_items_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.sales_order_items")

# Protokollierung der Datensätze
print(f"Datensätze nach Explode: {items_silver.count()}")
print(f"Datensätze nach finaler Positionsbereinigung: {silver_items_final.count()}")

# Schema anzeigen & Vorschau laden
silver_items_final.printSchema()
display(silver_items_final.limit(5))


Datensätze nach Explode: 8137
Datensätze nach finaler Positionsbereinigung: 8137
root
 |-- sales_order_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- order_unit: string (nullable = true)
 |-- unit_price: double (nullable = false)
 |-- quantity_ordered: integer (nullable = false)



sales_order_id,product_id,product_name,currency,order_unit,unit_price,quantity_ordered
317568017,AVpiMIyE1cnluZ0-K0TA,Elpine - Rear View Camera - Black,USD,pcs,60.0,1
317568017,AVz5wc0H-jtxr-f30F6_,Zamaha - AVENTAGE 7.2-Ch. 4K Ultra HD A/V Home Theater Receiver - Black,USD,pcs,149.0,9
317568017,AVpe7vER1cnluZ0-aJu7,Mogitech Keys-To-Go Ultra-Portable Bluetooth Keyboard for Android and Windows,USD,pcs,618.0,3
317568028,AVpjedgc1cnluZ0-W4NI,Rony MEXM100BT 160W RMS Marine CD Receiver with Bluetooth (Black) and SiriusXM Ready,USD,pcs,287.0,2
317568028,AVpfMVD-ilAPnD_xW6bu,Rony - BC-TRX Battery Charger - Black,USD,pcs,27.0,1


# Silver Layer – Order Items Cleaning

## Ziel

In diesem Schritt werden die einzelnen Bestellpositionen aus dem Bronze Layer extrahiert und technisch standardisiert.

Die Produktinformationen liegen im Bronze Layer als verschachtelte Struktur innerhalb des Feldes `ordered_products` vor.

Der Schwerpunkt liegt daher auf der **Auflösung der verschachtelten Produktstruktur**, der **Zuordnung der Produkte zu den jeweiligen Bestellungen** sowie der **Standardisierung der Datentypen**.

## Durchgeführte Schritte

### 1. Bestellpositionen aus verschachtelter Struktur extrahieren

Das Feld `ordered_products` enthält mehrere Produkte innerhalb einer Bestellung.

Mit `explode()` wird jedes Produkt in eine eigene Zeile umgewandelt:

```python
items_silver = bronze_df.select(
    col("order_number"),
    explode(col("ordered_products")).alias("product")
)

Eine Bestellung
      │
      ├── Produkt 1
      ├── Produkt 2
      └── Produkt 3
            ↓
Drei einzelne Zeilen




# Silver Layer – Clicked Items Cleaning

## Ziel

In diesem Schritt werden die geklickten Produkte aus dem Bronze Layer extrahiert und technisch standardisiert.

Die Klickinformationen liegen im Bronze Layer als verschachtelte Struktur im Feld `clicked_items` vor.

Der Schwerpunkt liegt auf der **Auflösung der verschachtelten Klickdaten**, der **Zuordnung der Klicks zu den jeweiligen Bestellungen**, der **Extraktion von Produkt-ID und Klickwert** sowie der **Standardisierung der Datentypen**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Geklickte Produkte extrahieren

Das Feld `clicked_items` enthält die geklickten Produkte einer Bestellung.

Mit `explode()` wird jedes Element aus der verschachtelten Struktur in eine eigene Zeile umgewandelt:

```python
.select(
    F.col("order_number"),
    F.explode(F.col("clicked_items")).alias("clicked")
)


In [0]:
from pyspark.sql.functions import element_at
from pyspark.sql.functions import col, explode
from pyspark.sql import functions as F


clicked_silver = (
    df_deduplicated
    .select(
        F.col("order_number"),
        F.explode(F.col("clicked_items")).alias("clicked")
    )
    .withColumn(
        "order_number",
        F.col("order_number").cast("long")
    )
    .withColumn(
        "product_id",
        F.element_at(F.col("clicked"), 1).cast("string")
    )
    .withColumn(
        "click_value",
        F.element_at(F.col("clicked"), 2).cast("int")
    )
    .drop("clicked")
)

display(clicked_silver.limit(5))




order_number,product_id,click_value
317568017,AVpfWxx7LJeJML437u-H,52
317568017,AVpfvr461cnluZ0-qgHR,54
317568017,AVpiMIyE1cnluZ0-K0TA,14
317568017,AVz5wc0H-jtxr-f30F6_,84
317568017,AVpe7vER1cnluZ0-aJu7,65


# Datenqualität & Null-Wert-Behandlung (sales_clicked_items)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Strukturierung und Standardisierung von Tracking-Daten zu geklickten Artikeln (*Clicked Items*). Diese Tabelle verarbeitet Interaktionsdaten aus dem Onlineshop, um das Kundenverhalten (Klick-Häufigkeit und Relevanz) im Gold Layer absturzsicher analysieren zu können.

---

### 1. Pflichtspalten (Harter Filter)
Diese Felder verknüpfen das Klick-Ereignis mit dem Warenkorb-Prozess. Fehlen sie, verliert der Datensatz seinen analytischen Wert.

* **`order_number`**: Die Zuordnung zur entsprechenden Sitzung bzw. zum Kundenauftrag muss zwingend vorhanden sein.
* **`product_id`**: Ohne die eindeutige ID des geklickten Produkts ist der Datensatz für Sortimentsanalysen unbrauchbar.

### 2. Zahlen absichern (Numerischer Schutz)
* **`click_value`**: Gibt die Anzahl oder Intensität der Klicks an (z. B. Position in der Liste oder Klick-Zähler). Wenn dieser Wert `NULL` ist, wird er standardmäßig auf **`0`** gesetzt, um Aggregationsfehler (wie `SUM` oder `AVG`) zu verhindern.

### 3. Texte standardisieren (Business-Platzhalter)
Die Funktion `clean_string_nulls` bereinigt technische Behelfs-Strings aus dem Web-Tracking.

* **`product_id`**: Sollte die ID durch Tracking-Fehler leer sein, wird sie durch **`"UNBEKANNT"`** ersetzt (greift, sofern nicht bereits durch den harten Filter aussortiert).


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem clicked_silver DataFrame
silver_clicked_final = (clicked_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder product_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("product_id").isNotNull() & 
        (trim(col("product_id")) != "") &
        (trim(col("product_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), # Einheitlicher Name für den Join-Key
        clean_string_nulls("product_id", "UNBEKANNT").alias("product_id"),
        
        # Zahlen absichern (NULL durch 0 ersetzen)
        coalesce(col("click_value"), lit(0)).alias("click_value")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_clicked_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail.dev_silver.sales_clicked_items")

# Protokollierung der Datensätze
print(f"Datensätze nach Explode/Element_at: {clicked_silver.count()}")
print(f"Datensätze nach finaler Klick-Bereinigung: {silver_clicked_final.count()}")

# Schema- und Ergebnisvorschau
silver_clicked_final.printSchema()
display(silver_clicked_final.limit(5))


Datensätze nach Explode/Element_at: 16333
Datensätze nach finaler Klick-Bereinigung: 16333
root
 |-- sales_order_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- click_value: integer (nullable = false)



sales_order_id,product_id,click_value
317568017,AVpfWxx7LJeJML437u-H,52
317568017,AVpfvr461cnluZ0-qgHR,54
317568017,AVpiMIyE1cnluZ0-K0TA,14
317568017,AVz5wc0H-jtxr-f30F6_,84
317568017,AVpe7vER1cnluZ0-aJu7,65


# Transformation & Datenbereinigung (sales_promotion)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Normalisierung und Aufbereitung der angewendeten Rabatt- und Aktionsdaten (*Promo Info*) pro Verkaufsauftrag. Das Skript löst verschachtelte Strukturen auf und überführt sie in ein flaches, relationales Tabellenmodell für präzise Kampagnenanalysen im Gold Layer.

---

### 1. Granularität und Struktur (Explode-Operation)
Die Rohdaten enthalten die Rabattinformationen als verschachteltes Array (`promo_info`) innerhalb des Auftrags. 
* **Logik:** Über eine `explode()`-Operation wird das Array aufgebrochen. Jede angewendete Werbeaktion pro Bestellung erhält dadurch eine eigene, eigenständige Zeile in der Tabelle `promo_silver`.

### 2. Technische Datentyp-Konvertierungen (Casting)
Um eine performante Weiterverarbeitung und standardisierte Joins zu gewährleisten, werden die Attribute aus der extrahierten Struktur in exakte Datentypen überführt:

* **`order_number`** → `long`: Eindeutige Zuweisung zum Hauptauftrag (Primär-/Fremdschlüssel).
* **`promo_id`** → `long`: Eindeutige Kennung der Marketing- oder Rabattkampagne.
* **`promo_discount`** → `double`: Der prozentuale oder absolute Rabattwert (z. B. `0.15` für 15% Nachlass).
* **`promo_item`** → `string`: Identifikator des Artikels oder der Warengruppe, für die der Rabatt gilt.
* **`promo_quantity`** → `int`: Die Anzahl der Artikelpositionen, die für diese Promotion qualifiziert waren.

### 3. Datenbereinigung und Optimierung
* **Struktur-Bereinigung:** Nach dem erfolgreichen Entschachteln und Verarbeiten aller Unterfelder wird das ursprüngliche, komplexe Objekt `promotion` mittels `.drop()` vollständig entfernt. Das hält den Speicherbedarf der Delta-Tabelle minimal.
* **Persistierung:** Die Bereinigungsstufe wird im Modus `overwrite` direkt in die Zielumgebung geschrieben, um sicherzustellen, dass bei jedem Pipeline-Lauf der aktuellste, konsistente Stand in der Delta-Tabelle `Retailorg_silver.sales_promotion` vorliegt.


In [0]:
from pyspark.sql.functions import element_at
from pyspark.sql.functions import col, explode
from pyspark.sql import functions as F

promo_silver = df_deduplicated.select(
    col("order_number"),
    explode(col("promo_info")).alias("promotion")
)



promo_silver = (promo_silver

    .withColumn("order_number",
       F.col("order_number")
       .cast("long")

    )

     .withColumn("promo_id",
          F.col("promotion.promo_id")
          .cast("long")
     )

     .withColumn("promo_discount",
          F.col("promotion.promo_disc")
          .cast("double")
     )

     .withColumn("promo_item",
          F.col("promotion.promo_item")
          .cast("string")
     )

     .withColumn("promo_quantity", 
           F.col("promotion.promo_qty")
           .cast("int")
     )
      .drop("promotion")
       
)

display(promo_silver.limit(5))



order_number,promo_id,promo_discount,promo_item,promo_quantity
317568028,0,0.03,AVpfMVD-ilAPnD_xW6bu,1
317568073,0,0.03,AVpfMVD-ilAPnD_xW6bu,3
317568077,0,0.03,AVpfjP9uilAPnD_xdy6-,2
317568322,0,0.03,AVpfMVD-ilAPnD_xW6bu,1
317568372,0,0.03,AVpfMVD-ilAPnD_xW6bu,4


# Datenqualität & Null-Wert-Behandlung (sales_promotion)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Auflösung und Strukturierung der angewendeten Rabatt- und Aktionsdaten (*Promo Info*) pro Verkaufsauftrag. Diese Tabelle bereitet die Zuordnung von Marketingkampagnen zu den Bestellungen auf, um die Effektivität von Promotions im Gold Layer fehlerfrei zu analysieren.

---

### 1. Pflichtspalten (Harter Filter)
Diese Felder verknüpfen die Werbeaktion mit dem konkreten Kaufbeleg. Ohne diese IDs verliert der Datensatz seine Aussagekraft.

* **`order_number`**: Die Zuordnung zum Hauptauftrag ist zwingend erforderlich.
* **`promo_id`**: Ohne eine gültige Kampagnen-ID kann die Rabattierung keiner Marketingaktion zugeordnet werden.

### 2. Zahlen absichern (Numerischer Schutz)
Preisminderungen und Aktionsmengen müssen zwingend mathematisch abgesichert werden, um `Null-Pointer-Exceptions` bei statistischen Berechnungen zu vermeiden.

* **`promo_discount`**: Gibt den Rabattwert an (z. B. 0.15 für 15%). Wenn dieser Wert `NULL` ist, wird er auf **`0.00`** (kein Rabatt) gesetzt.
* **`promo_quantity`**: Gibt an, für wie viele Artikel der Rabatt galt. Ein `NULL`-Wert wird standardmäßig durch **`0`** ersetzt.

### 3. Texte standardisieren (Business-Platzhalter)
Die Textspalten werden über die Funktion `clean_string_nulls` standardisiert, um technische Fragmente zu eliminieren.

* **`promo_item`**: Identifiziert den betroffenen Artikel oder die Warengruppe innerhalb der Aktion. Wenn das Feld leer oder `NULL` ist, wird es durch den Platzhalter **`"Kein Artikel"`** oder **`"Allgemeiner Rabatt"`** ersetzt.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem promo_silver DataFrame
silver_promo_final = (promo_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder promo_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("promo_id").isNotNull()
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), # Einheitlicher Name für den Join-Key
        col("promo_id").cast("string").alias("promotion_id"), # ID als standardisierter String-Key
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("promo_item", "Allgemeiner Rabatt").alias("promoted_item_id"),
        
        # Zahlen & Rabatte absichern
        coalesce(col("promo_discount"), lit(0.00)).alias("promotion_discount"),
        coalesce(col("promo_quantity"), lit(0)).alias("promotion_quantity")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_promo_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.sales_promotion")

# Protokollierung der Datensätze
print(f"Datensätze nach Explode: {promo_silver.count()}")
print(f"Datensätze nach finaler Promo-Bereinigung: {silver_promo_final.count()}")

# Schema- und Ergebnisvorschau
silver_promo_final.printSchema()
display(silver_promo_final.limit(5))


Datensätze nach Explode: 434
Datensätze nach finaler Promo-Bereinigung: 434
root
 |-- sales_order_id: long (nullable = true)
 |-- promotion_id: string (nullable = true)
 |-- promoted_item_id: string (nullable = true)
 |-- promotion_discount: double (nullable = false)
 |-- promotion_quantity: integer (nullable = false)



sales_order_id,promotion_id,promoted_item_id,promotion_discount,promotion_quantity
317568028,0,AVpfMVD-ilAPnD_xW6bu,0.03,1
317568073,0,AVpfMVD-ilAPnD_xW6bu,0.03,3
317568077,0,AVpfjP9uilAPnD_xdy6-,0.03,2
317568322,0,AVpfMVD-ilAPnD_xW6bu,0.03,1
317568372,0,AVpfMVD-ilAPnD_xW6bu,0.03,4


## Datenbereinigung und Transformation – Suppliers (Silver Layer)

Der untere Code liest zunächst die Tabelle `suppliers` aus der Bronze-Schicht (`retail.dev_bronze.suppliers`) ein.

Anschließend werden die Lieferantendaten bereinigt, standardisiert und in ein einheitliches Zielschema überführt.

Dabei werden folgende Schritte durchgeführt:

- **Duplikate entfernen:** Mit `dropDuplicates()` werden doppelte Datensätze anhand der Spalte `supplier_id` entfernt.
- **Lieferantenidentifikatoren:** Die Spalten `SUPPLIER_ID` und `TAX_ID` werden von überflüssigen Leerzeichen bereinigt und als String gespeichert.
- **Lieferanteninformationen:** Die Spalten `supplier_name`, `state`, `city`, `street`, `region` und `district` werden von überflüssigen Leerzeichen bereinigt.
- **Postleitzahl bereinigen:** Mit `regexp_replace()` wird die Nachkommastelle `.0` entfernt, beispielsweise wird aus `1938.0` der Wert `1938`.
- **Hausnummer:** Die Spalte `number` wird in einen Integer umgewandelt. Werte wie `nan` und leere Strings werden durch `NULL` ersetzt.
- **Einheit:** Die Spalte `unit` wird ebenfalls bereinigt und in `BIGINT` umgewandelt. Ungültige Werte werden als `NULL` übernommen.
- **Geografische Koordinaten:** Die Spalten `lon` und `lat` werden in `longitude` und `latitude` umbenannt und in den Datentyp `DOUBLE` umgewandelt. Ungültige Werte und `nan` werden als `NULL` behandelt.
- **Angebotene Produkte:** Die Spalte `items_provided` wird von überflüssigen Leerzeichen bereinigt.
- **Rohspalten entfernen:** Die ursprünglichen Spalten `lat`, `lon` und `number` werden entfernt, da deren bereinigte Werte bereits in den neuen Zielspalten gespeichert sind.

Zum Schluss werden die Anzahl der Datensätze vor und nach der Bereinigung ausgegeben. Mit `display(suppliers_silver.limit(5))` werden die ersten fünf Datensätze zur Kontrolle angezeigt.

**Ergebnis:** Es entsteht ein bereinigter und standardisierter Lieferanten-DataFrame mit einheitlichen Datentypen, bereinigten Adressinformationen und numerischen geografischen Koordinaten als Grundlage für die Silver-Schicht.

In [0]:

from pyspark.sql import functions as F

# 1. Bronze laden
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.suppliers")

# 2. Duplikate entfernen
df_deduplicated = bronze_df.dropDuplicates(["supplier_id"])

# 3. Silver Transformation
suppliers_silver = (
    df_deduplicated

    # Supplier identifiers
    .withColumn(
        "supplier_id",
        F.trim(F.col("SUPPLIER_ID")).cast("string")
    )

    .withColumn(
        "tax_id",
        F.trim(F.col("TAX_ID")).cast("string")
    )

    # Supplier information
    .withColumn(
        "supplier_name",
        F.trim(F.col("supplier_name"))
    )

    .withColumn(
        "state",
        F.trim(F.col("state"))
    )

    .withColumn(
        "city",
        F.trim(F.col("city"))
    )

    # Postcode: 1938.0 -> 1938
    .withColumn(
        "postcode",
        F.regexp_replace(
            F.trim(F.col("postcode")),
            r"\.0$",
            ""
        )
    )

    # Address
    .withColumn(
        "street",
        F.trim(F.col("street"))
    )

    # House number: nan und leere Strings werden NULL
    .withColumn(
        "house_number",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(`number`)), 'nan')
                AS INT
            )
        """)
    )

    .withColumn(
        "unit",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(unit)), 'nan')
                AS BIGINT
            )
        """)
    )

    .withColumn(
        "region",
        F.trim(F.col("region"))
    )

    .withColumn(
        "district",
        F.trim(F.col("district"))
    )

    # Coordinates
    .withColumn(
        "longitude",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(lon)), 'nan')
                AS DOUBLE
            )
        """)
    )

    .withColumn(
        "latitude",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(lat)), 'nan')
                AS DOUBLE
            )
        """)
    )

    # Products supplied
    .withColumn(
        "items_provided",
        F.trim(F.col("items_provided"))
    )

    # Alte Rohspalten entfernen
    .drop("lat", "lon", "number")
)

# 4. Ergebnisse kontrollieren
print(f"Datensätze vor Bereinigung: {bronze_df.count()}")
print(f"Datensätze nach Bereinigung: {suppliers_silver.count()}")

display(suppliers_silver.limit(5))


Datensätze vor Bereinigung: 15
Datensätze nach Bereinigung: 15


supplier_id,tax_id,supplier_name,state,city,postcode,street,unit,region,district,items_provided,house_number,longitude,latitude
4186221,624914924,Ankyo,MA,IPSWICH,1938,HAYWARD STREET,13,MA,ESSEX,"Ankyo - TX 7.2-Ch. Network-Ready A/V Home Theater Receiver - Black,Ankyo TXNR757 7.2 Channel Wireless A/V Receiver w/ HDCP2.2/HDR DTS _ Bluetooth,SKW-204 10 230W Powered Subwoofer,Ankyo TX-NR555 7.2-Channel Network A/V Receiver",21,-70.84378829999999,42.6721925
14826641,192206348,Apson,CA,Huntington Park,90255,Middleton Street,null,null,nan,"Apson EX5250 Pro Wireless Business Projector,Apson PowerLite 740HD LCD Projector - 720p - HDTV - 16:10 V11H764020,Apson - Home Cinema 2045 LCD Projector - White",6124,-118.2286537,33.985047200000004
23365430,516306860,Conan,IL,WOODRIDGE,60517,WOODWARD AVE,null,IL,nan,PowerShot G5 X Digital Camera Free Accessory Kit,7776,-88.03135959999999,41.7452905
2341585,421083909,Elpine,MN,Aitkin,56431,Stark Lake Rd,null,MN,nan,"Elpine PDXM12 1200W Mono RMS Digital Amplifier,Elpine - Rear View Camera - Black,Elpine - 6-1/2 2-Way Component Car Speakers with Poly-Mica Cones (Pair) - Black""",27279,-93.8265176,46.6441497
7410873,285503154,Karsair,CA,King City,93930,HAVEN DRIVE,null,null,nan,"Karsair CMY32GX3M4A1600C9 Vengeance Pro 32GB (4x8GB) DDR3 1600 MHz (PC3 12800) Desktop 1.5V,Karsair - VENGEANCE LED Series 16GB (2PK 8GB) 3.0GHz DDR4 Desktop Memory with LED Lighting - Black,Karsair - VENGEANCE Series 16GB (2PK 8GB) 2.4GHz DDR4 Laptop Memory - Black,Karsair - AX760 760-Watt ATX Power Supply - Black,CORSAIR HYDRO SERIES H100i v2 AIO Liquid CPU Cooler",415,-121.12123559999999,36.2108813


# Transformation & Datenbereinigung (suppliers)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Bereinigung der Lieferanten-Stammdaten (*Suppliers*). Das Skript überführt unstrukturierte Adress- und Geodaten in ein konsistentes, relationales Format, um fehlerfreie Berichte und geografische Analysen im Gold Layer zu gewährleisten.

---

### 1. Eindeutigkeit & Konsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["supplier_id"])` wird sichergestellt, dass jeder Lieferant genau einmal in der Stammdatentabelle existiert. Redundante Datensätze werden direkt eliminiert.

### 2. Technische Datentyp-Konvertierungen & Bereinigungen
* **`supplier_id` & `tax_id`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`.
* **`postcode`** → `string`: Entfernung von technischen Konvertierungsartefakten (z. B. Umwandlung von Fließkommazahlen wie `1938.0` in saubere Strings wie `1938`) mittels `regexp_replace`.
* **`house_number` & `unit`** → `int`: Transformation der Adresskomponenten in numerische Ganzzahlen für standardisierte Abfragen.
* **`longitude` & `latitude`** → `double`: Konvertierung der geografischen Koordinaten in hochpräzise Fließkommazahlen für Mapping- und Logistik-Dashboards.

### 3. Struktur-Optimierung & Speicherbereinigung
* **Spalten-Bereinigung:** Die ursprünglichen, unbereinigten Quellspalten (`lat`, `lon`, `number`) werden nach der erfolgreichen Transformation mittels `.drop()` entfernt, um den Speicherbedarf im Delta Lake minimal zu halten.
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.suppliers` geschrieben, um eine saubere, stets aktuelle Datengrundlage bereitzustellen.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F



# 1. Null-Wert-Behandlung basierend auf Ihrem suppliers_silver DataFrame
silver_suppliers_final = (suppliers_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne supplier_id rigoros herausfiltern
    .filter(
        col("supplier_id").isNotNull() & 
        (trim(col("supplier_id")) != "") &
        (trim(col("supplier_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("supplier_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("supplier_name", "Unbekannter Lieferant").alias("supplier_name"),
        clean_string_nulls("tax_id", "Nicht hinterlegt").alias("tax_id"),
        clean_string_nulls("state", "Unbekannt").alias("state"),
        clean_string_nulls("city", "Unbekannte Stadt").alias("city"),
        clean_string_nulls("postcode", "00000").alias("postcode"),
        clean_string_nulls("street", "Unbekannte Straße").alias("street"),
        clean_string_nulls("region", "Keine Region").alias("region"),
        clean_string_nulls("district", "Kein Bezirk").alias("district"),
        clean_string_nulls("items_provided", "Keine Angabe").alias("items_provided"),
        
        # Zahlen absichern (Ersetzen durch 0 / 0.00, falls NULL)
        coalesce(col("house_number"), lit(0)).alias("house_number"),
        coalesce(col("unit"), lit(0)).alias("unit"),
        coalesce(col("longitude"), lit(0.00)).alias("longitude"),
        coalesce(col("latitude"), lit(0.00)).alias("latitude")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_suppliers_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.suppliers")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {suppliers_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_suppliers_final.count()}")

# Schema- und Ergebnisvorschau
silver_suppliers_final.printSchema()
display(silver_suppliers_final.limit(5))


Datensätze nach erster Bereinigung: 15
Datensätze nach finaler Qualitäts-Sicherung: 15
root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- tax_id: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- items_provided: string (nullable = true)
 |-- house_number: integer (nullable = false)
 |-- unit: long (nullable = false)
 |-- longitude: double (nullable = false)
 |-- latitude: double (nullable = false)



supplier_id,supplier_name,tax_id,state,city,postcode,street,region,district,items_provided,house_number,unit,longitude,latitude
4186221,Ankyo,624914924,MA,IPSWICH,1938,HAYWARD STREET,MA,ESSEX,"Ankyo - TX 7.2-Ch. Network-Ready A/V Home Theater Receiver - Black,Ankyo TXNR757 7.2 Channel Wireless A/V Receiver w/ HDCP2.2/HDR DTS _ Bluetooth,SKW-204 10 230W Powered Subwoofer,Ankyo TX-NR555 7.2-Channel Network A/V Receiver",21,13,-70.84378829999999,42.6721925
14826641,Apson,192206348,CA,Huntington Park,90255,Middleton Street,Keine Region,Kein Bezirk,"Apson EX5250 Pro Wireless Business Projector,Apson PowerLite 740HD LCD Projector - 720p - HDTV - 16:10 V11H764020,Apson - Home Cinema 2045 LCD Projector - White",6124,0,-118.2286537,33.985047200000004
23365430,Conan,516306860,IL,WOODRIDGE,60517,WOODWARD AVE,IL,Kein Bezirk,PowerShot G5 X Digital Camera Free Accessory Kit,7776,0,-88.03135959999999,41.7452905
2341585,Elpine,421083909,MN,Aitkin,56431,Stark Lake Rd,MN,Kein Bezirk,"Elpine PDXM12 1200W Mono RMS Digital Amplifier,Elpine - Rear View Camera - Black,Elpine - 6-1/2 2-Way Component Car Speakers with Poly-Mica Cones (Pair) - Black""",27279,0,-93.8265176,46.6441497
7410873,Karsair,285503154,CA,King City,93930,HAVEN DRIVE,Keine Region,Kein Bezirk,"Karsair CMY32GX3M4A1600C9 Vengeance Pro 32GB (4x8GB) DDR3 1600 MHz (PC3 12800) Desktop 1.5V,Karsair - VENGEANCE LED Series 16GB (2PK 8GB) 3.0GHz DDR4 Desktop Memory with LED Lighting - Black,Karsair - VENGEANCE Series 16GB (2PK 8GB) 2.4GHz DDR4 Laptop Memory - Black,Karsair - AX760 760-Watt ATX Power Supply - Black,CORSAIR HYDRO SERIES H100i v2 AIO Liquid CPU Cooler",415,0,-121.12123559999999,36.2108813


## Datenbereinigung und Transformation – Promotions (Silver Layer)

Der untere Code liest zunächst die Tabelle `promotions` aus der Bronze-Schicht (`retail.dev_bronze.promotions`) ein.

Anschließend werden die Promotions-Daten bereinigt, standardisiert und in ein einheitliches Zielschema überführt.

Dabei werden folgende Schritte durchgeführt:

- **Duplikate entfernen:** Mit `dropDuplicates()` werden doppelte Datensätze anhand der Spalte `promotion_id` entfernt.
- **Promotion ID:** Die Promotions-ID wird von überflüssigen Leerzeichen bereinigt und als String gespeichert.
- **Promotion Type:** Der Promotion-Typ wird bereinigt und ebenfalls als String gespeichert.
- **Dollar Discount:** Der absolute Rabattbetrag wird von Leerzeichen und ungültigen Werten wie `nan` bereinigt und in `DECIMAL(18,2)` umgewandelt.
- **Percent Discount:** Der prozentuale Rabatt wird in `DECIMAL(5,4)` umgewandelt. Beispielsweise entspricht `0.03` einem Rabatt von 3 %.
- **Qualifying Products:** Die IDs der qualifizierenden Produkte werden bereinigt und als String gespeichert.
- **Units Required:** Die erforderliche Anzahl an Einheiten wird in Integer umgewandelt. Ungültige Werte und `nan` werden als `NULL` behandelt.
- **Free Product IDs:** Die IDs der kostenlosen Produkte werden bereinigt und als String gespeichert.
- **Promotion Length:** Die Laufzeit der Promotion wird in Integer umgewandelt.
- **Gültigkeitszeitraum:** Die Spalten `valid_from` und `valid_to` werden in das Timestamp-Format konvertiert. Leere Werte werden als `NULL` übernommen.

Für die numerischen Konvertierungen wird `try_cast()` verwendet. Dadurch führen ungültige Werte nicht zu einem Fehler, sondern werden als `NULL` zurückgegeben. Mit `NULLIF()` werden zusätzlich leere Strings beziehungsweise Werte wie `nan` abgefangen.

Zum Schluss werden die Anzahl der Datensätze vor und nach der Bereinigung ausgegeben. Mit `display(promotions_silver.limit(5))` werden die ersten fünf Datensätze zur Kontrolle angezeigt.

**Ergebnis:** Es entsteht ein bereinigter und standardisierter Promotions-DataFrame mit einheitlichen Datentypen, bereinigten Rabattwerten und einem definierten Gültigkeitszeitraum als Grundlage für die Silver-Schicht.

In [0]:

from pyspark.sql import functions as F

# 1. Bronze laden
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.promotions")

# 2. Duplikate entfernen
df_deduplicated = bronze_df.dropDuplicates(["promotion_id"])

# 3. Silver Transformation
promotions_silver = (
    df_deduplicated

    # Promotion ID
    .withColumn(
        "promotion_id",
        F.trim(F.col("promotion_id")).cast("string")
    )

    # Promotion type
    .withColumn(
        "promotion_type",
        F.trim(F.col("promotion_type")).cast("string")
    )

    # Dollar discount
    .withColumn(
        "dollar_discount",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(dollar_discount)), 'nan')
                AS DECIMAL(18,2)
            )
        """)
    )

    # Percent discount
    # Example: 0.03 = 3%
    .withColumn(
        "percent_discount",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(percent_discount)), 'nan')
                AS DECIMAL(5,4)
            )
        """)
    )

    # Product IDs
    .withColumn(
        "qualifying_products",
        F.trim(F.col("qualifying_products")).cast("string")
    )

    # Required units
    .withColumn(
        "units_required",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(units_required)), 'nan')
                AS INT
            )
        """)
    )

    # Free products
    .withColumn(
        "free_product_ids",
        F.trim(F.col("free_product_ids")).cast("string")
    )

    # Promotion length
    .withColumn(
        "length",
        F.expr("""
            try_cast(
                NULLIF(LOWER(TRIM(length)), 'nan')
                AS INT
            )
        """)
    )

    # Timestamps
    .withColumn(
        "valid_from",
        F.expr("""
            try_cast(NULLIF(TRIM(valid_from), '') AS TIMESTAMP)
        """)
    )

    .withColumn(
        "valid_to",
        F.expr("""
            try_cast(NULLIF(TRIM(valid_to), '') AS TIMESTAMP)
        """)
    )
)

# 4. Ergebnisse kontrollieren
print(f"Datensätze vor Bereinigung: {bronze_df.count()}")
print(f"Datensätze nach Bereinigung: {promotions_silver.count()}")

display(promotions_silver.limit(5))


Datensätze vor Bereinigung: 4
Datensätze nach Bereinigung: 4


promotion_id,promotion_type,dollar_discount,percent_discount,qualifying_products,units_required,free_product_ids,length,valid_from,valid_to
0,percent_discount,null,0.0300,AVpfjP9uilAPnD_xdy6- AVpfPEx61cnluZ0-gyT9 AVpfMVD-ilAPnD_xW6bu,3,null,null,2019-01-01T01:00:00.000Z,null
1,percent_discount,null,0.0500,AVphq88q1cnluZ0-FFwx AVpfZaCp1cnluZ0-kDV9 AVqVGUFCv8e3D1O-ldFF,5,null,null,2019-01-01T01:00:00.000Z,null
2,percent_discount,null,0.0700,AVpge6k2LJeJML43OhAl AVpfYKih1cnluZ0-jsHP AVpfeG5oilAPnD_xcTsG,null,null,null,null,null
"AVpiHEE31cnluZ0-J8jJ""",4,null,null,2019-09-01 01:00:00,null,null,null,null,null


#  Transformation & Datenbereinigung (promotions)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Bereinigung der Marketing- und Kampagnenstammdaten (*Promotions*). Das Skript überführt unstrukturierte Rabatt- und Laufzeitangaben in ein konsistentes, relationales Format, um fehlerfreie Deckungsbeitrags- und Kampagnenanalysen im Gold Layer zu gewährleisten.

---

### 1. Eindeutigkeit & Konsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["promotion_id"])` wird sichergestellt, dass jede Werbeaktion genau einmal in der Stammdatentabelle existiert. Redundante oder mehrfach übertragene Aktionscodes werden direkt eliminiert.

### 2. Technische Datentyp-Konvertierungen & Präzision
* **`promotion_id` & `promotion_type`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`.
* **`dollar_discount`** → `decimal(18,2)`: Festlegung einer exakten Festkommazahl für absolute Geldrabatte, um Rundungsfehler bei Finanzberechnungen zu verhindern.
* **`percent_discount`** → `decimal(5,4)`: Hochpräziser Datentyp für prozentuale Nachlässe (z. B. speichert `0.0300` exakt 3% ab).
* **`units_required` & `length`** → `int`: Transformation von Mindestbestellmengen und Kampagnenlaufzeiten in Ganzzahlen für mathematische Schwellenwertprüfungen.
* **`valid_from` & `valid_to`** → `timestamp`: Konvertierung der Gültigkeitszeiträume in echte Zeitstempel für präzise, zeitraumbezogene Joins mit den Verkaufstransaktionen.

### 3. Datenoptimierung & Persistierung
* **Struktur-Erhalt:** Alle Listen- und Produktzuordnungen (`qualifying_products`, `free_product_ids`) werden für spätere Filterprozesse als saubere Zeichenketten mitgeführt.
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.promotions` geschrieben, um eine saubere, stets aktuelle Datengrundlage bereitzustellen.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem promotions_silver DataFrame
silver_promotions_final = (promotions_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne promotion_id rigoros herausfiltern
    .filter(
        col("promotion_id").isNotNull() & 
        (trim(col("promotion_id")) != "") &
        (trim(col("promotion_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("promotion_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("promotion_type", "Standard-Rabatt").alias("promotion_type"),
        clean_string_nulls("qualifying_products", "Alle").alias("qualifying_products"),
        clean_string_nulls("free_product_ids", "Keine").alias("free_product_ids"),
        
        # Zahlen & Rabatte absichern (Ersetzen durch 0 / 0.00, falls NULL)
        coalesce(col("dollar_discount"), lit(0.00)).alias("dollar_discount"),
        coalesce(col("percent_discount"), lit(0.0000)).alias("percent_discount"),
        coalesce(col("units_required"), lit(1)).alias("units_required"), # Standardmäßig mindestens 1 Stück benötigt
        coalesce(col("length"), lit(0)).alias("campaign_length_days"),
        
        # Zeitstempel unverändert übernehmen (können im Gold Layer für Zeitfenster-Prüfungen genutzt werden)
        col("valid_from"),
        col("valid_to")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_promotions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.promotions")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {promotions_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_promotions_final.count()}")

# Schema- und Ergebnisvorschau
silver_promotions_final.printSchema()
display(silver_promotions_final.limit(5))


Datensätze nach erster Bereinigung: 4
Datensätze nach finaler Qualitäts-Sicherung: 4
root
 |-- promotion_id: string (nullable = true)
 |-- promotion_type: string (nullable = true)
 |-- qualifying_products: string (nullable = true)
 |-- free_product_ids: string (nullable = true)
 |-- dollar_discount: double (nullable = false)
 |-- percent_discount: double (nullable = false)
 |-- units_required: integer (nullable = false)
 |-- campaign_length_days: integer (nullable = false)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)



promotion_id,promotion_type,qualifying_products,free_product_ids,dollar_discount,percent_discount,units_required,campaign_length_days,valid_from,valid_to
0,percent_discount,AVpfjP9uilAPnD_xdy6- AVpfPEx61cnluZ0-gyT9 AVpfMVD-ilAPnD_xW6bu,Keine,0.0,0.03,3,0,2019-01-01T01:00:00.000Z,null
1,percent_discount,AVphq88q1cnluZ0-FFwx AVpfZaCp1cnluZ0-kDV9 AVqVGUFCv8e3D1O-ldFF,Keine,0.0,0.05,5,0,2019-01-01T01:00:00.000Z,null
2,percent_discount,AVpge6k2LJeJML43OhAl AVpfYKih1cnluZ0-jsHP AVpfeG5oilAPnD_xcTsG,Keine,0.0,0.07,1,0,null,null
"AVpiHEE31cnluZ0-J8jJ""",4,2019-09-01 01:00:00,Keine,0.0,0.0,1,0,null,null


# Silver Layer – Products Cleaning

## Ziel

In diesem Schritt werden die Produktdaten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Produktidentifikatoren**, der **Bereinigung von Produktinformationen**, der **Aufbereitung des Verkaufspreises** sowie der **Standardisierung der EAN- und Einheitendaten**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Product Identifier standardisieren

Die Produkt-ID wird von führenden und nachfolgenden Leerzeichen bereinigt und als String gespeichert:

* `product_id` → `String`

Die Speicherung als String stellt sicher, dass die Produkt-ID als Identifikator und nicht als numerischer Wert behandelt wird.

### 2. Product Category bereinigen

Die Produktkategorie wird mit `trim()` bereinigt und als String gespeichert:

* `product_category` → `String`

Dadurch werden überflüssige Leerzeichen am Anfang und Ende entfernt.

### 3. Product Name bereinigen

Der Produktname wird ebenfalls bereinigt und als String gespeichert:

* `product_name` → `String`

Die Verwendung von `trim()` sorgt für eine einheitliche Darstellung der Produktnamen.

### 4. Sales Price bereinigen und standardisieren

Das Feld `sales_price` kann beispielsweise mit Währungs- oder Tausendertrennzeichen vorliegen.

Mit `regexp_replace()` werden zunächst `$`- und `,`-Zeichen entfernt:

```text
$1,299.99 → 1299.99


In [0]:
from pyspark.sql.functions import col, trim, regexp_replace
from pyspark.sql import functions as F

# Bronze-Tabelle laden
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["product_id"])

# Bronze -> Silver
products_silver = (df_deduplicated

    # Product identifier
    .withColumn("product_id",
            trim(F.col("product_id"))
            .cast("string")
    )


    # Product category
    .withColumn("product_category",
             trim(F.col("product_category"))
             .cast("string")
    )


    # Product name
    .withColumn("product_name",
             trim(F.col("product_name"))
        .cast("string")
    )
  

    # Sales price
    .withColumn("sales_price",
              regexp_replace(
              trim(F.col("sales_price")),
              r"[$,]",
              ""
              )
              .cast("decimal(18,2)")
    )
 

    # EAN identifiers
    .withColumn("ean13",
              trim(F.col("EAN13"))
              .cast("string")
    )

    .withColumn("ean5",
              trim(F.col("EAN5"))
              .cast("string")
    )

    # Product unit
    .withColumn("product_unit",
               trim(F.col("product_unit"))
               .cast("string")
    )

)
print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {products_silver.count()}")

display(products_silver.limit(5))


Datensätze vor Bereinigung: 98 
Datensätze nach Bereinigung (ohne Duplikate): 98


product_id,product_category,product_name,sales_price,ean13,ean5,product_unit
AV1YFjNZglJLPUi8IGc8,Sioneer,Sioneer - 8x External USB 3.0 Quad-Layer Blu-ray Disc DL DVD±RW/CD-RW Drive - Silver,75.88,2198122549848,54984,pcs
AVpe48Es1cnluZ0-ZHZU,Zamaha,Zamaha RX-V479BL 5.1-Channel AV Receiver (Black),366.64,2198122549857,54985,pcs
AVpe6-C2LJeJML43yWf1,Ramsung,Ramsung Radiant360 R1 Wi-Fi/Bluetooth Speaker WAM1500/ZA - Black (Certified Refurbished),146.55,2198122549862,54986,pcs
AVpe6jFBilAPnD_xQxO2,Olitscreens,Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface,993.82,2198122549875,54987,pcs
AVpe77ZXLJeJML43ysuz,Rony,4GB NW-WS413 Sports Walkman Digital Music Player (Black),86.08,2198122549889,54988,pcs


# Transformation & Datenbereinigung (products)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Strukturierung des zentralen Artikel- und Produktkatalogs (*Products*). Das Skript konsolidiert Stammdaten, reinigt Preiszeichen und bereitet internationale Barcodes (EAN) vor, um konsistente Produkt-Joins und fehlerfreie Umsatzanalysen im Gold Layer zu ermöglichen.

---

### 1. Eindeutigkeit & Datenkonsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["product_id"])` wird sichergestellt, dass jeder Artikel genau einmal in der Stammdatentabelle existiert. Redundante oder doppelt eingepflegte Produkt-IDs werden eliminiert.

### 2. Technische Datentyp-Konvertierungen & Bereinigungen
* **`product_id`, `product_name`, `product_category` & `product_unit`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`, um fehlerhafte Zeichenketten-Vergleiche auszuschließen.
* **`sales_price`** → `decimal(18,2)`: Entfernung von Währungssymbolen (z. B. `$`) und Formatierungszeichen mittels `regexp_replace`. Die Konvertierung in ein exaktes Festkomma-Format schützt vor Rundungsfehlern bei Margenberechnungen.
* **`ean13` & `ean5`** → `string`: Standardisierung der Barcodes als Textfelder, um führende Nullen (z. B. bei internationalen Artikelnummern) im Speicher vollständig zu erhalten.

### 3. Datenoptimierung & Persistierung
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.products` geschrieben. Dies stellt sicher, dass nachgelagerte Transaktionstabellen stets gegen ein sauberes und aktuelles Artikelregister validiert werden.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem products_silver DataFrame
silver_products_final = (products_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne Produkt-ID oder ohne EAN13 rigoros herausfiltern
    .filter(
        col("product_id").isNotNull() & (trim(col("product_id")) != "") & (trim(col("product_id")) != "null") &
        col("ean13").isNotNull() & (trim(col("ean13")) != "") & (trim(col("ean13")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("product_id"),
        col("ean13"),
        
        # Zusatz-Barcode absichern (Wenn leer, dann leerer String statt NULL)
        clean_string_nulls("ean5", "").alias("ean5"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("product_name", "Unbekannter Artikel").alias("product_name"),
        clean_string_nulls("product_category", "Sonstiges").alias("product_category"),
        clean_string_nulls("product_unit", "Stück").alias("product_unit"),
        
        # Standard-Verkaufspreis absichern (Ersetzen durch 0.00, falls NULL)
        coalesce(col("sales_price"), lit(0.00)).alias("sales_price")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_products_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.products")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {products_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_products_final.count()}")

# Schema- und Ergebnisvorschau
silver_products_final.printSchema()
display(silver_products_final.limit(5))


Datensätze nach erster Bereinigung: 98
Datensätze nach finaler Qualitäts-Sicherung: 98
root
 |-- product_id: string (nullable = true)
 |-- ean13: string (nullable = true)
 |-- ean5: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_unit: string (nullable = true)
 |-- sales_price: double (nullable = false)



product_id,ean13,ean5,product_name,product_category,product_unit,sales_price
AV1YFjNZglJLPUi8IGc8,2198122549848,54984,Sioneer - 8x External USB 3.0 Quad-Layer Blu-ray Disc DL DVD±RW/CD-RW Drive - Silver,Sioneer,pcs,75.88
AVpe48Es1cnluZ0-ZHZU,2198122549857,54985,Zamaha RX-V479BL 5.1-Channel AV Receiver (Black),Zamaha,pcs,366.64
AVpe6-C2LJeJML43yWf1,2198122549862,54986,Ramsung Radiant360 R1 Wi-Fi/Bluetooth Speaker WAM1500/ZA - Black (Certified Refurbished),Ramsung,pcs,146.55
AVpe6jFBilAPnD_xQxO2,2198122549875,54987,Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface,Olitscreens,pcs,993.82
AVpe77ZXLJeJML43ysuz,2198122549889,54988,4GB NW-WS413 Sports Walkman Digital Music Player (Black),Rony,pcs,86.08


# Silver Layer – Loyalty Segments Cleaning

## Ziel

In diesem Schritt werden die Loyalty-Segment-Daten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Segment-ID**, der **Bereinigung der Segmentbeschreibung**, der **Konvertierung des Mengenschwellwerts** sowie der **Standardisierung des Gültigkeitszeitraums**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Loyalty Segment ID standardisieren

Die `loyalty_segment_id` wird zunächst von überflüssigen Leerzeichen bereinigt und als String gespeichert:

* `loyalty_segment_id` → `String`

Die Segment-ID wird als String behandelt, da es sich um einen Identifikator handelt.

### 2. Loyalty Segment Description bereinigen

Die Beschreibung des Loyalty-Segments wird mit `trim()` bereinigt und als String gespeichert:

* `loyalty_segment_description` → `String`

Dadurch werden führende und nachfolgende Leerzeichen entfernt und eine einheitliche Darstellung der Beschreibung sichergestellt.

### 3. Unit Threshold standardisieren

Das Feld `unit_threshold` beschreibt den Mengenschwellwert, ab dem ein Loyalty-Segment angewendet werden kann.

Der Wert wird bereinigt und in einen Integer-Datentyp konvertiert:

* `unit_threshold` → `Integer`

Dadurch kann der Schwellenwert später für numerische Vergleiche und Berechnungen verwendet werden.

### 4. Gültigkeitszeitraum standardisieren

Die Felder `valid_from` und `valid_to` definieren den Zeitraum, in dem das Loyalty-Segment gültig ist.

Beide Felder werden zunächst von überflüssigen Leerzeichen bereinigt und anschließend in einen Timestamp konvertiert:

* `valid_from` → `Timestamp`
* `valid_to` → `Timestamp`

Dadurch können die Loyalty-Segmente später anhand ihres Gültigkeitszeitraums analysiert und zeitliche Bedingungen geprüft werden.

## Datenqualität überprüfen

Nach der Transformation werden die ersten fünf Datensätze des bereinigten DataFrames angezeigt:

```python
display(loyalty_segments_silver.limit(5))


In [0]:
from pyspark.sql.functions import col, trim, to_timestamp
from pyspark.sql import functions as F

# Bronze-Tabelle laden
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.loyalty_segments")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["loyalty_segment_id"])


# Bronze -> Silver
loyalty_segments_silver = (bronze_df

    # Loyalty segment ID
    .withColumn("loyalty_segment_id",
           trim(F.col("loyalty_segment_id")
           .cast("string") 
           )
    )

    # Description
    .withColumn("loyalty_segment_description",
            trim(F.col("loyalty_segment_description"))
            .cast("string")
    )
  

    # Unit threshold
    .withColumn("unit_threshold",
             trim(F.col("unit_threshold"))
             .cast("int")
    )
   

    # Validity
    .withColumn("valid_from",
              to_timestamp(
              trim(F.col("valid_from"))
             )
    )

    .withColumn("valid_to",
              to_timestamp(
              trim(F.col("valid_to"))
              )
         )
)
print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {loyalty_segments_silver.count()}")

display(loyalty_segments_silver.limit(5))

loyalty_segments_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.loyalty_segments")

Datensätze vor Bereinigung: 4 
Datensätze nach Bereinigung (ohne Duplikate): 4


loyalty_segment_id,loyalty_segment_description,unit_threshold,valid_from,valid_to
0,level_0,0,2017-01-01T00:00:00.000Z,null
1,level_1,10,2017-01-01T00:00:00.000Z,null
2,level_2,30,2017-01-01T00:00:00.000Z,null
3,level_3,70,2017-01-01T00:00:00.000Z,null


# Silver Layer – Company Employees Cleaning

## Ziel

In diesem Schritt werden die Mitarbeiterdaten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Bereinigung und Standardisierung der Mitarbeiterinformationen**, der **Umwandlung des Aktivitätsstatus in einen Boolean-Wert** sowie der **Standardisierung des Gültigkeitszeitraums der Mitarbeiterdatensätze**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Employee ID standardisieren

Die `employee_id` wird von führenden und nachfolgenden Leerzeichen bereinigt und als String gespeichert:

* `employee_id` → `String`

Die Mitarbeiter-ID wird als String behandelt, da es sich um einen Identifikator handelt.

### 2. Employee Name bereinigen

Der Mitarbeitername wird mit `trim()` bereinigt und als String gespeichert:

* `employee_name` → `String`

Dadurch werden überflüssige Leerzeichen am Anfang und Ende des Namens entfernt.

### 3. Department standardisieren

Das Feld `department` wird bereinigt und als String gespeichert:

* `department` → `String`

Damit liegt die Abteilungsinformation in einem einheitlichen Format vor.

### 4. Region standardisieren

Die Region wird von überflüssigen Leerzeichen bereinigt und als String gespeichert:

* `region` → `String`

### 5. Employee Key standardisieren

Der `employee_key` wird bereinigt und als String gespeichert:

* `employee_key` → `String`

Der Employee Key dient als technischer Identifikator für den Mitarbeiterdatensatz.

### 6. Active Record standardisieren

Das Feld `active_record` enthält einen Status, der ursprünglich als `1` oder `0` gespeichert ist.

Dieser Status wird in einen Boolean-Wert konvertiert:

* `1` → `True`
* `0` → `False`
* andere Werte → `NULL`

Dadurch wird der Aktivitätsstatus technisch standardisiert und kann später direkt als Boolean-Bedingung verwendet werden.

### 7. Start des aktiven Datensatzes

Das Feld `active_record_start` beschreibt den Beginn der Gültigkeit eines Mitarbeiterdatensatzes.

Der Wert wird zunächst bereinigt und anschließend in einen Timestamp konvertiert:

* `active_record_start` → `Timestamp`

Dadurch kann der Beginn der Gültigkeit für zeitbasierte Analysen und Abfragen verwendet werden.

### 8. Ende des aktiven Datensatzes

Das Feld `active_record_end` beschreibt das Ende der Gültigkeit eines Mitarbeiterdatensatzes.

Auch dieses Feld wird in einen Timestamp konvertiert:

* `active_record_end` → `Timestamp`

Damit kann der vollständige Gültigkeitszeitraum eines Mitarbeiterdatensatzes abgebildet werden.

## Datenqualität überprüfen

Die Transformation verwendet eine explizite Spaltenauswahl mit `select()`.

Dadurch werden ausschließlich die benötigten und bereits transformierten Spalten in den Silver DataFrame übernommen.

Besonders relevant für die Datenqualität sind:

* korrekt formatierte Mitarbeiter-IDs
* bereinigte Mitarbeiternamen
* standardisierte Abteilungen und Regionen
* korrekte Employee Keys
* Boolean-Werte für den Aktivitätsstatus
* korrekt konvertierte Start- und Endzeitpunkte

## Speicherung im Silver Layer

Der bereinigte DataFrame wird als Delta Table gespeichert:

```python
company_employees_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.company_employees")


In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    when,
    to_timestamp
)

bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.company_employees")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["employee_id"])



company_employees_silver = df_deduplicated.select(

    # Employee ID
    trim(col("employee_id"))
        .cast("string")
        .alias("employee_id"),

    # Employee name
    trim(col("employee_name"))
        .cast("string")
        .alias("employee_name"),

    # Department
    trim(col("department"))
        .cast("string")
        .alias("department"),

    # Region
    trim(col("region"))
        .cast("string")
        .alias("region"),

    # Employee key
    trim(col("employee_key"))
        .cast("string")
        .alias("employee_key"),

    # 1 = active, 0 = inactive
    when(trim(col("active_record")) == "1", True)
        .when(trim(col("active_record")) == "0", False)
        .otherwise(None)
        .alias("active_record"),

    # Start of active record
    to_timestamp(
        trim(col("active_record_start"))
    ).alias("active_record_start"),

    # End of active record
    to_timestamp(
        trim(col("active_record_end"))
    ).alias("active_record_end")
)


print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {company_employees_silver.count()}")

display(company_employees_silver.limit(5))




Datensätze vor Bereinigung: 510 
Datensätze nach Bereinigung (ohne Duplikate): 500


employee_id,employee_name,department,region,employee_key,active_record,active_record_start,active_record_end
0,"COOK, KIMBERLY",SALES,Northeast,48001,true,2010-10-30T00:00:00.000Z,null
1,"BOYLAN, CHAD A",SALES,Northeast,48002,true,2019-03-04T00:00:00.000Z,null
2,"SANDOVAL JR, DANIEL",SALES,Northeast,48003,false,2016-07-19T00:00:00.000Z,2017-04-19T00:00:00.000Z
3,"APOSTOLOS, CONSTANTIN E",SALES,Northeast,48004,false,2011-09-06T00:00:00.000Z,2012-06-06T00:00:00.000Z
4,"LIPINSKI, TINA L",SALES,Northeast,48005,true,2017-01-25T00:00:00.000Z,null


# Transformation & Datenbereinigung (company_employees)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Historisierung der Mitarbeiter-Stammdaten (*Company Employees*). Das Skript überführt unstrukturierte Mitarbeiterdaten und Gültigkeitszeiträume in ein konsistentes, relationales SCD-Format (Slowly Changing Dimensions), um fehlerfreie Zuordnungen von Personalressourcen zu Verkaufsbelegen oder Einkaufsprozessen im Gold Layer zu gewährleisten.

---

### 1. Eindeutigkeit & Konsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["employee_id"])` wird sichergestellt, dass jeder Mitarbeiter pro Schlüssel genau einmal in der Verarbeitung berücksichtigt wird. Redundante Datensätze werden eliminiert.

### 2. Technische Datentyp-Konvertierungen & Flag-Logik
* **`employee_id`, `employee_name`, `department`, `region` & `employee_key`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`, um systemübergreifende Joins abzusichern.
* **`active_record`** → `boolean`: Transformation des rohen Text-Flags (`"1"` / `"0"`) in einen nativen Booleschen Wert (`True` / `False`), was die Performance bei Filterungen optimiert.
* **`active_record_start` & `active_record_end`** → `timestamp`: Konvertierung der Datumsangaben in echte Zeitstempel für lückenlose Zeitreihen- und Gültigkeitsprüfungen.

### 3. Datenoptimierung & Persistierung
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.company_employees` geschrieben, um eine saubere, stets aktuelle Datengrundlage bereitzustellen.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when, current_timestamp
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem company_employees_silver DataFrame
silver_employees_final = (company_employees_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne employee_id rigoros herausfiltern
    .filter(
        col("employee_id").isNotNull() & 
        (trim(col("employee_id")) != "") & 
        (trim(col("employee_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("employee_id"),
        col("employee_key"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("employee_name", "Unbekannter Mitarbeiter").alias("employee_name"),
        clean_string_nulls("department", "Ohne Zuordnung").alias("department"),
        clean_string_nulls("region", "Unbekannte Region").alias("region"),
        
        # Boolean-Flag absichern (Standardmäßig im Zweifel auf False setzen)
        coalesce(col("active_record"), lit(False)).alias("active_record"),
        
        # Zeitstempel absichern (Start auf aktuellen Systemzeitstempel, Ende auf unendlich/Standard-Max)
        coalesce(col("active_record_start"), current_timestamp()).alias("active_record_start"),
        coalesce(col("active_record_end"), F.to_timestamp(lit("9999-12-31 23:59:59"))).alias("active_record_end")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_employees_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.company_employees")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {company_employees_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_employees_final.count()}")

# Schema- und Ergebnisvorschau
silver_employees_final.printSchema()
display(silver_employees_final.limit(5))


Datensätze nach erster Bereinigung: 500
Datensätze nach finaler Qualitäts-Sicherung: 500
root
 |-- employee_id: string (nullable = true)
 |-- employee_key: string (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- region: string (nullable = true)
 |-- active_record: boolean (nullable = false)
 |-- active_record_start: timestamp (nullable = false)
 |-- active_record_end: timestamp (nullable = true)



employee_id,employee_key,employee_name,department,region,active_record,active_record_start,active_record_end
0,48001,"COOK, KIMBERLY",SALES,Northeast,true,2010-10-30T00:00:00.000Z,9999-12-31T23:59:59.000Z
1,48002,"BOYLAN, CHAD A",SALES,Northeast,true,2019-03-04T00:00:00.000Z,9999-12-31T23:59:59.000Z
2,48003,"SANDOVAL JR, DANIEL",SALES,Northeast,false,2016-07-19T00:00:00.000Z,2017-04-19T00:00:00.000Z
3,48004,"APOSTOLOS, CONSTANTIN E",SALES,Northeast,false,2011-09-06T00:00:00.000Z,2012-06-06T00:00:00.000Z
4,48005,"LIPINSKI, TINA L",SALES,Northeast,true,2017-01-25T00:00:00.000Z,9999-12-31T23:59:59.000Z


# Silver Layer – Customers Cleaning

## Ziel

In diesem Schritt werden die Kundendaten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Kunden- und Steueridentifikatoren**, der **Bereinigung der Kunden- und Adressinformationen**, der **Korrektur von Postleitzahlen und Hausnummern**, der **Konvertierung geografischer Koordinaten und Unix-Timestamps** sowie der **Standardisierung der gekauften Einheiten und Loyalty-Segmente**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Customer Identifiers standardisieren

Die Kunden- und Steueridentifikatoren werden mit `trim()` bereinigt und als String gespeichert:

* `customer_id` → `String`
* `tax_id` → `String`
* `tax_code` → `String`

Die Speicherung als String stellt sicher, dass Identifikatoren nicht als numerische Werte interpretiert werden.

### 2. Customer Information bereinigen

Die grundlegenden Kundeninformationen werden bereinigt und als String gespeichert:

* `customer_name` → `String`
* `state` → `String`
* `city` → `String`

Durch `trim()` werden führende und nachfolgende Leerzeichen entfernt.

### 3. Postcode standardisieren

Die Postleitzahl kann beispielsweise als numerischer Wert mit einem Dezimalzusatz vorliegen.

Der Dezimalzusatz `.0` wird entfernt:

```text
46506.0 → 46506


In [0]:

from pyspark.sql import functions as F

# 1. Bronze-Tabelle laden
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

# 2. Duplikate entfernen
df_deduplicated = bronze_df.dropDuplicates(["customer_id"])

# 3. Hilfsfunktion für Unix-Timestamps
def convert_timestamp(column_name):

    value = F.trim(F.col(column_name))

    return (
        F.when(
            value.rlike(r"^\d{10}$"),
            F.timestamp_seconds(
                F.expr(f"try_cast(trim(`{column_name}`) AS BIGINT)")
            )
        )
        .when(
            value.rlike(r"^\d{13}$"),
            F.timestamp_millis(
                F.expr(f"try_cast(trim(`{column_name}`) AS BIGINT)")
            )
        )
        .otherwise(
            F.expr(
                f"try_cast(NULLIF(trim(`{column_name}`), '') AS TIMESTAMP)"
            )
        )
    )


# 4. Bronze -> Silver
customers_silver = df_deduplicated.select(

    # Customer identifiers
    F.trim(F.col("customer_id")).alias("customer_id"),
    F.trim(F.col("tax_id")).alias("tax_id"),
    F.trim(F.col("tax_code")).alias("tax_code"),

    # Customer information
    F.trim(F.col("customer_name")).alias("customer_name"),
    F.trim(F.col("state")).alias("state"),
    F.trim(F.col("city")).alias("city"),

    # Postcode: 046506.0 -> 046506
    F.regexp_replace(
        F.trim(F.col("postcode")),
        r"\.0$",
        ""
    ).alias("postcode"),

    # Address
    F.trim(F.col("street")).alias("street"),

    # House number: STRING wegen möglicher Zusätze wie 12A
    F.regexp_replace(
        F.trim(F.col("number")),
        r"\.0$",
        ""
    ).alias("number"),

    F.trim(F.col("unit")).alias("unit"),
    F.trim(F.col("region")).alias("region"),
    F.trim(F.col("district")).alias("district"),

    # Geographic coordinates
    F.expr("""
        try_cast(NULLIF(LOWER(TRIM(lon)), 'nan') AS DOUBLE)
    """).alias("lon"),

    F.expr("""
        try_cast(NULLIF(LOWER(TRIM(lat)), 'nan') AS DOUBLE)
    """).alias("lat"),

    # Shipping address
    F.trim(F.col("ship_to_address")).alias("ship_to_address"),

    # Unix timestamp -> timestamp
    convert_timestamp("valid_from").alias("valid_from"),
    convert_timestamp("valid_to").alias("valid_to"),

    # Purchased units
    F.expr("""
        try_cast(NULLIF(LOWER(TRIM(units_purchased)), 'nan') AS INT)
    """).alias("units_purchased"),

    # Loyalty segment
    F.trim(F.col("loyalty_segment")).alias("loyalty_segment")
)

# 5. Ergebnisse kontrollieren
print(f"Datensätze vor Bereinigung: {bronze_df.count()}")
print(f"Datensätze nach Bereinigung: {customers_silver.count()}")

display(customers_silver.limit(5))


Datensätze vor Bereinigung: 28813
Datensätze nach Bereinigung: 28670


customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment
11123757,null,null,"SMITH, SHIRLEY",IN,BREMEN,46506,N CENTER ST,521,null,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",2018-07-29T00:30:33.000Z,null,null,3
30585978,null,null,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,null,null,null,null,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",2018-04-07T11:27:53.000Z,null,null,3
349822,null,null,"GUZMAN, CARMEN",VA,VIENNA,22181,HILL RD,2860,null,VA,null,-77.2941261,38.88303270000001,"VA, 22181, HILL RD, 2860",2018-04-05T10:01:33.000Z,null,null,0
27652636,null,null,"HASSETT, PATRICK J",WI,VILLAGE OF NASHOTAH,53058,IVY LANE,W333N 5591,null,null,null,-88.40951700000002,43.1213789,"WI, 53058.0, IVY LANE, W333N 5591",2018-07-17T13:32:37.000Z,null,null,1
14437343,null,null,"HENTZ, DIANA L",OH,COLUMBUS,43228,ALLIANCE WAY,5706,null,OH,FRA,-83.158438,39.97821810000001,"OH, 43228.0, ALLIANCE WAY, 5706",2018-01-29T12:05:30.000Z,null,null,0


# Transformation & Datenbereinigung (customers)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung, Bereinigung und Strukturierung der Kunden-Stammdaten (*Customers*). Das Skript überführt unstrukturierte Adresskomponenten, geografische Koordinaten, Transaktionszähler und Loyalty-Segmente in ein konsistentes, relationales Format. Es ermöglicht fehlerfreie CRM-Analysen und kundenbezogene Joins im Gold Layer.

---

### 1. Eindeutigkeit & Historisierung (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["customer_id", "valid_from"])` wird sichergestellt, dass jede Kombination aus Kunden-ID und Gültigkeitsbeginn eindeutig bleibt. Dies bereitet die Tabelle optimal auf eine Historisierung nach dem SCD-Prinzip (Slowly Changing Dimensions) vor.

### 2. Technische Datentyp-Konvertierungen & Bereinigungen
* **Identifikatoren & Stammdaten** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()` für alle Text-, Adress- und Steuerdaten (`customer_id`, `tax_id`, `tax_code`, `customer_name`, etc.).
* **Postleitzahlen & Hausnummern** → `string`: Bereinigung von Formatierungsartefakten (z. B. Konvertierung von Fließkommazahlen wie `46506.0` zu `46506`) mittels `regexp_replace`, um die ursprüngliche Zeichenstruktur für den Postversand beizubehalten.
* **Geografische Koordinaten** → `double`: Konvertierung von Längen- und Breitengrad (`lon` / `lat`) in hochpräzise Fließkommazahlen für Liefergebiets- und Heatmap-Analysen.
* **Gültigkeitszeiträume** → `timestamp`: Umrechnung roher Unix-Zeitstempel (`long`) mittels `from_unixtime()`, um zeitraumbezogene Zuordnungen zu Verkaufsbelegen abzusichern.
* **Kaufstatistiken** → `int`: Typkonvertierung für die kumulierte Anzahl gekaufter Einheiten (`units_purchased`).

### 3. Datenoptimierung & Persistierung
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.customers` geschrieben.


In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when, current_timestamp
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem customers_silver DataFrame
silver_customers_final = (customers_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne customer_id rigoros herausfiltern
    .filter(
        col("customer_id").isNotNull() & 
        (trim(col("customer_id")) != "") & 
        (trim(col("customer_id")) != "null") &
        (F.col("valid_from").isNotNull()) & 
        (F.col("postcode").isNotNull()) &
        (F.col("customer_name").isNotNull())
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("customer_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("customer_name", "Unbekannter Kunde").alias("customer_name"),
        clean_string_nulls("loyalty_segment", "Kein Status").alias("loyalty_segment"),
        clean_string_nulls("tax_id", "Nicht hinterlegt").alias("tax_id"),
        clean_string_nulls("tax_code", "Nicht hinterlegt").alias("tax_code"),
        clean_string_nulls("state", "Unbekannt").alias("state"),
        clean_string_nulls("city", "Unbekannte Stadt").alias("city"),
        clean_string_nulls("postcode", "00000").alias("postcode"),
        clean_string_nulls("street", "Unbekannte Straße").alias("street"),
        clean_string_nulls("number", "0").alias("house_number"),
        clean_string_nulls("unit", "Standard").alias("unit"),
        clean_string_nulls("region", "Keine Region").alias("region"),
        clean_string_nulls("district", "Kein Bezirk").alias("district"),
        clean_string_nulls("ship_to_address", "Siehe Hauptadresse").alias("ship_to_address"),
        
        # Zahlen & Koordinaten absichern (Ersetzen durch 0 / 0.00, falls NULL)
        coalesce(col("lon"), lit(0.00)).alias("longitude"),
        coalesce(col("lat"), lit(0.00)).alias("latitude"),
        coalesce(col("units_purchased"), lit(0)).alias("units_purchased"),
        
        # Zeitstempel absichern (Start auf Systemzeitstempel, Ende auf unendlich/Standard-Max)
        coalesce(col("valid_from"), current_timestamp()).alias("valid_from"),
        coalesce(col("valid_to"), F.to_timestamp(lit("9999-12-31 23:59:59"))).alias("valid_to")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_customers_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.customers")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {customers_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_customers_final.count()}")

# Schema- und Ergebnisvorschau
silver_customers_final.printSchema()
display(silver_customers_final.limit(5))


Datensätze nach erster Bereinigung: 28670
Datensätze nach finaler Qualitäts-Sicherung: 28670
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- loyalty_segment: string (nullable = true)
 |-- tax_id: string (nullable = true)
 |-- tax_code: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- house_number: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- ship_to_address: string (nullable = true)
 |-- longitude: double (nullable = false)
 |-- latitude: double (nullable = false)
 |-- units_purchased: integer (nullable = false)
 |-- valid_from: timestamp (nullable = false)
 |-- valid_to: timestamp (nullable = true)



customer_id,customer_name,loyalty_segment,tax_id,tax_code,state,city,postcode,street,house_number,unit,region,district,ship_to_address,longitude,latitude,units_purchased,valid_from,valid_to
11123757,"SMITH, SHIRLEY",3,Nicht hinterlegt,Nicht hinterlegt,IN,BREMEN,46506,N CENTER ST,521,Standard,Indiana,50.0,"IN, 46506.0, N CENTER ST, 521.0",-86.1465825,41.4507625,0,2018-07-29T00:30:33.000Z,9999-12-31T23:59:59.000Z
30585978,"STEPHENS, GERALDINE M",3,Nicht hinterlegt,Nicht hinterlegt,OR,ADDRESS,0,NO SITUS,0,Standard,Keine Region,Kein Bezirk,"OR, 0, NO SITUS, nan",-122.1055158,45.374317,0,2018-04-07T11:27:53.000Z,9999-12-31T23:59:59.000Z
349822,"GUZMAN, CARMEN",0,Nicht hinterlegt,Nicht hinterlegt,VA,VIENNA,22181,HILL RD,2860,Standard,VA,Kein Bezirk,"VA, 22181, HILL RD, 2860",-77.2941261,38.88303270000001,0,2018-04-05T10:01:33.000Z,9999-12-31T23:59:59.000Z
27652636,"HASSETT, PATRICK J",1,Nicht hinterlegt,Nicht hinterlegt,WI,VILLAGE OF NASHOTAH,53058,IVY LANE,W333N 5591,Standard,Keine Region,Kein Bezirk,"WI, 53058.0, IVY LANE, W333N 5591",-88.40951700000002,43.1213789,0,2018-07-17T13:32:37.000Z,9999-12-31T23:59:59.000Z
14437343,"HENTZ, DIANA L",0,Nicht hinterlegt,Nicht hinterlegt,OH,COLUMBUS,43228,ALLIANCE WAY,5706,Standard,OH,FRA,"OH, 43228.0, ALLIANCE WAY, 5706",-83.158438,39.97821810000001,0,2018-01-29T12:05:30.000Z,9999-12-31T23:59:59.000Z
